In [1]:
# Import libraries and other configurations
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

sns.set_theme(style="whitegrid")

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 121

In [2]:
# Load the cleaned dataset
df = pd.read_csv(
    "../data/processed/cleaned_donor_data.csv",
    dtype={"donor_postal_code": "str"}
)

df.columns.tolist()

['donor_unique_id',
 'donor_postal_code',
 'donor_age',
 'gender_identity',
 'is_member_flag',
 'is_alumnus_flag',
 'is_parent_flag',
 'has_involvement_flag',
 'preferred_address_type',
 'has_email_flag',
 'consecutive_donor_years',
 'last_fiscal_year_donation',
 'donation_2_fiscal_years_ago',
 'donation_3_fiscal_years_ago',
 'donation_4_fiscal_years_ago',
 'donation_5_fiscal_years_ago',
 'current_fiscal_year_donation',
 'cumulative_donation_amount',
 'donor_indicator_flag']

In [3]:
# Verify dataset loaded correctly
print(f"Dataset Shape: {df.shape}")
df.sample(10, random_state=RANDOM_STATE)

Dataset Shape: (34403, 19)


,donor_unique_id,donor_postal_code,donor_age,gender_identity,is_member_flag,is_alumnus_flag,is_parent_flag,has_involvement_flag,preferred_address_type,has_email_flag,consecutive_donor_years,last_fiscal_year_donation,donation_2_fiscal_years_ago,donation_3_fiscal_years_ago,donation_4_fiscal_years_ago,donation_5_fiscal_years_ago,current_fiscal_year_donation,cumulative_donation_amount,donor_indicator_flag
33267,33371,42301.0,46,Female,0,0,0,0,Home,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
26595,26683,45620.0,42,Female,0,0,0,0,Home,0,1,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
16653,16704,33427.0,28,Female,0,0,0,0,Home,0,0,0.00,120.00,0.00,0.00,0.00,0.00,120.00,1
21428,21498,54131.0,33,Female,0,0,0,0,Home,0,0,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1
16873,16924,90265.0,33,Male,0,1,0,0,Home,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
19919,19982,54459.0,32,Female,0,0,0,0,Home,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
15478,15527,64686.0,42,Male,0,0,0,0,Home,0,8,0.00,0.00,10.00,0.00,0.00,0.00,"9,680.00",1
591,596,12067.0,42,Female,0,0,1,0,Home,0,0,0.00,100.00,1.00,0.00,0.00,0.00,101.00,1
1474,1483,90265.0,42,Female,0,1,0,0,Home,1,0,0.00,0.00,0.00,0.00,0.00,0.00,479.00,1
22846,22922,45856.0,36,Female,0,0,0,0,Home,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0


In [4]:
# Verify dataset data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34403 entries, 0 to 34402
Data columns (total 19 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   donor_unique_id               34403 non-null  int64  
 1   donor_postal_code             34312 non-null  str    
 2   donor_age                     34403 non-null  int64  
 3   gender_identity               33912 non-null  str    
 4   is_member_flag                34403 non-null  int64  
 5   is_alumnus_flag               34403 non-null  int64  
 6   is_parent_flag                34403 non-null  int64  
 7   has_involvement_flag          34403 non-null  int64  
 8   preferred_address_type        30370 non-null  str    
 9   has_email_flag                34403 non-null  int64  
 10  consecutive_donor_years       34403 non-null  int64  
 11  last_fiscal_year_donation     34403 non-null  float64
 12  donation_2_fiscal_years_ago   34403 non-null  float64
 13  donation_3_f

In [5]:
# Data types summary
dtype_summary = (
    df.dtypes
      .astype(str)
      .value_counts()
      .reset_index()
      .style.hide(axis="index")
)

dtype_summary.columns = ["Data Type", "Count"]
dtype_summary

index,count
int64,9
float64,7
str,3


In [6]:
# Verify dataset missing values
missing_summary = (
    df.isna()
      .sum()
      .to_frame("Missing Count")
)

missing_summary["Missing %"] = (
    missing_summary["Missing Count"] / len(df) * 100
)

missing_summary = (
    missing_summary
      .sort_values("Missing Count", ascending=False)
)

missing_summary

,Missing Count,Missing %
preferred_address_type,4033,11.72
gender_identity,491,1.43
donor_postal_code,91,0.26
donor_unique_id,0,0.00
last_fiscal_year_donation,0,0.00
cumulative_donation_amount,0,0.00
current_fiscal_year_donation,0,0.00
donation_5_fiscal_years_ago,0,0.00
donation_4_fiscal_years_ago,0,0.00
donation_3_fiscal_years_ago,0,0.00


## Dataset Overview
The finalized cleaned analytical dataset was successfully loaded and verified for feature engineering. The dataset contains 34,403 donor records and 19 features, consistent with the dataset used throughout Phase 2. Data types and missing values match expectations from the data cleaning process, and all required libraries were successfully imported. The only change made was explicitly defining `donor_postal_code` as a string. This was made because it was interpreting postal codes as a float and not a string.

In [7]:
# Define target-related column names
TARGET_SOURCE_COLUMN = "current_fiscal_year_donation"
TARGET_COLUMN = "target_current_fiscal_year_donor_flag"

# Validate the target source before creating the binary target
assert pd.api.types.is_numeric_dtype(df[TARGET_SOURCE_COLUMN]), (
    f"{TARGET_SOURCE_COLUMN} must contain numeric values."
)

# Check for missing values
missing_target_values = df[TARGET_SOURCE_COLUMN].isna().sum()
print(f"Missing target-source values: {missing_target_values:,}")

assert missing_target_values == 0, (
    f"{TARGET_SOURCE_COLUMN} contains "
    f"{missing_target_values:,} missing values."
)

# Check for negative numbers
negative_target_values = (df[TARGET_SOURCE_COLUMN] < 0).sum()
print(f"Negative target-source values: {negative_target_values:,}")

assert negative_target_values == 0, (
    f"{TARGET_SOURCE_COLUMN} contains "
    f"{negative_target_values:,} negative values."
)

Missing target-source values: 0
Negative target-source values: 0


In [8]:
# Create the binary prediction target
df[TARGET_COLUMN] = (
    df[TARGET_SOURCE_COLUMN] > 0
).astype("int8")

print(f"Target column created: {TARGET_COLUMN}")

Target column created: target_current_fiscal_year_donor_flag


In [9]:
# Summarize the target distribution
target_summary = (
    df[TARGET_COLUMN]
    .value_counts()
    .sort_index()
    .rename_axis(TARGET_COLUMN)
    .reset_index(name="count")
)

target_summary["percentage"] = (
    target_summary["count"] / len(df) * 100
).round(2)

target_summary["target_label"] = target_summary[TARGET_COLUMN].map(
    {
        0: "Did not donate",
        1: "Donated",
    }
)

target_summary = target_summary[
    [
        TARGET_COLUMN,
        "target_label",
        "count",
        "percentage",
    ]
]

display(target_summary.style.hide(axis="index").format({"percentage": "{:.2f}%", "count": "{:,}"}))

target_current_fiscal_year_donor_flag,target_label,count,percentage
0,Did not donate,"32,499",94.47%
1,Donated,"1,904",5.53%


In [10]:
# Confirm that the target contains only binary values and matches its source column
expected_target_values = {0, 1}
actual_target_values = set(df[TARGET_COLUMN].unique())

assert actual_target_values == expected_target_values, (
    f"Unexpected target values found: {actual_target_values}"
)

# Confirm that the target matches its source column
target_mismatch_count = (
    df[TARGET_COLUMN]
    != (df[TARGET_SOURCE_COLUMN] > 0).astype("int8")
).sum()

assert target_mismatch_count == 0, (
    f"Found {target_mismatch_count:,} target-definition mismatches."
)

print("Target validation passed.")

Target validation passed.


## Prediction Target and Time Window

The classification target is `target_current_fiscal_year_donor_flag`, which identifies whether an individual donated during the current fiscal year.

The target is derived from `current_fiscal_year_donation`:

- `0`: The individual donated $0 during the current fiscal year
  
- `1`: The individual donated more than $0 during the current fiscal year

The target distribution shows that 1,904 individuals, or 5.53% of the dataset, donated during the current fiscal year. The remaining 32,499 individuals, or 94.47%, did not donate. This indicates a substantial class imbalance that will need to be considered during model evaluation and training.

The intended prediction setup uses the five completed fiscal years before the current fiscal year as the historical observation window. The current fiscal year serves as the outcome period.

The model will therefore use information available before the beginning of the current fiscal year to predict whether an individual donates during that fiscal year.

The original `current_fiscal_year_donation` column will remain in the working dataset temporarily for validation, but it must not be included in the predictor matrix because it directly determines the target.

Until additional documentation is available:

* `cumulative_donation_amount` will not be used directly because it may include current-fiscal-year donations
* `donor_indicator_flag` will not be used as a predictor until its calculation timing and logic are confirmed
* `consecutive_donor_years` will not be used until it is confirmed that it excludes current-fiscal-year activity
* Demographic and engagement fields will be treated as provisionally available
* The exact fiscal-year dates and dataset extraction date should be confirmed before the final model is deployed

The timing and leakage status of all source variables will be evaluated during the formal leakage audit.


In [11]:
# Define leakage audit classifications
VALID_LEAKAGE_CLASSIFICATIONS = {
    "Safe predictor",
    "Potential leakage",
    "Identifier",
    "Target",
    "Excluded feature",
}

# Document the leakage decision for every column
leakage_audit = pd.DataFrame(
    [
        {
            "column": "donor_unique_id",
            "classification": "Identifier",
            "model_treatment": "Exclude from predictors",
            "reason": (
                "Unique record identifier with no meaningful predictive value."
            ),
            "required_action": (
                "Retain only for record tracking and exported predictions."
            ),
        },
        {
            "column": "donor_postal_code",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Geographic information should be available before the "
                "prediction period and does not directly reveal the target."
            ),
            "required_action": (
                "Evaluate cardinality, privacy, and fairness before modeling."
            ),
        },
        {
            "column": "donor_age",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Demographic information does not directly reveal whether "
                "the individual donated during the target period."
            ),
            "required_action": (
                "Document that the exact age reference date is unavailable."
            ),
        },
        {
            "column": "gender_identity",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Demographic information does not directly reveal the "
                "current-fiscal-year donation outcome."
            ),
            "required_action": (
                "Consolidate unknown and missing categories before modeling."
            ),
        },
        {
            "column": "is_member_flag",
            "classification": "Excluded feature",
            "model_treatment": "Exclude from predictors",
            "reason": (
                "The field is constant in the cleaned dataset and therefore "
                "provides no predictive variation."
            ),
            "required_action": "Exclude from the modeling feature set.",
        },
        {
            "column": "is_alumnus_flag",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Alumni status is a relatively stable relationship attribute "
                "and does not directly reveal the target."
            ),
            "required_action": (
                "Retain unless later documentation identifies a timing issue."
            ),
        },
        {
            "column": "is_parent_flag",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Parent status is a relatively stable relationship attribute "
                "and does not directly reveal the target."
            ),
            "required_action": (
                "Retain unless later documentation identifies a timing issue."
            ),
        },
        {
            "column": "has_involvement_flag",
            "classification": "Potential leakage",
            "model_treatment": "Exclude pending verification",
            "reason": (
                "The snapshot date is unknown, so the field may include "
                "involvement established during the target period."
            ),
            "required_action": (
                "Confirm that the value was available at the prediction cutoff."
            ),
        },
        {
            "column": "preferred_address_type",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Contact-preference information does not directly reveal "
                "current-fiscal-year donation behavior."
            ),
            "required_action": (
                "Standardize categories and handle missing values before modeling."
            ),
        },
        {
            "column": "has_email_flag",
            "classification": "Potential leakage",
            "model_treatment": "Exclude pending verification",
            "reason": (
                "The field may reflect contact information added or updated "
                "during the target period."
            ),
            "required_action": (
                "Confirm that the value was available at the prediction cutoff."
            ),
        },
        {
            "column": "consecutive_donor_years",
            "classification": "Potential leakage",
            "model_treatment": "Exclude pending verification",
            "reason": (
                "The count may include donation activity from the current "
                "fiscal year."
            ),
            "required_action": (
                "Confirm the calculation period or reconstruct it from "
                "historical donation columns."
            ),
        },
        {
            "column": "last_fiscal_year_donation",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Represents donation behavior from the completed fiscal year "
                "before the target period."
            ),
            "required_action": "Retain as a historical predictor.",
        },
        {
            "column": "donation_2_fiscal_years_ago",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Represents donation behavior from before the target period."
            ),
            "required_action": "Retain as a historical predictor.",
        },
        {
            "column": "donation_3_fiscal_years_ago",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Represents donation behavior from before the target period."
            ),
            "required_action": "Retain as a historical predictor.",
        },
        {
            "column": "donation_4_fiscal_years_ago",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Represents donation behavior from before the target period."
            ),
            "required_action": "Retain as a historical predictor.",
        },
        {
            "column": "donation_5_fiscal_years_ago",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Represents donation behavior from before the target period."
            ),
            "required_action": "Retain as a historical predictor.",
        },
        {
            "column": "current_fiscal_year_donation",
            "classification": "Excluded feature",
            "model_treatment": "Exclude from predictors",
            "reason": (
                "Directly determines the binary modeling target and would "
                "cause target leakage if used as a predictor."
            ),
            "required_action": (
                "Retain only for target construction and validation."
            ),
        },
        {
            "column": "cumulative_donation_amount",
            "classification": "Excluded feature",
            "model_treatment": "Exclude or reconstruct",
            "reason": (
                "The cumulative value may include the current-fiscal-year "
                "donation and therefore reveal target-period information."
            ),
            "required_action": (
                "Replace with a historical value calculated only from "
                "pre-cutoff donation data."
            ),
        },
        {
            "column": "donor_indicator_flag",
            "classification": "Potential leakage",
            "model_treatment": "Exclude pending verification",
            "reason": (
                "The calculation logic and timing are unknown and may include "
                "current-fiscal-year donation behavior."
            ),
            "required_action": (
                "Confirm the source logic and reference date before use."
            ),
        },
        {
            "column": "target_current_fiscal_year_donor_flag",
            "classification": "Target",
            "model_treatment": "Use as modeling target",
            "reason": (
                "Binary outcome created from whether "
                "current_fiscal_year_donation is greater than zero."
            ),
            "required_action": (
                "Use as the target variable and never include it among predictors."
            ),
        },
    ]
)

display(leakage_audit.style.hide(axis="index"))

column,classification,model_treatment,reason,required_action
donor_unique_id,Identifier,Exclude from predictors,Unique record identifier with no meaningful predictive value.,Retain only for record tracking and exported predictions.
donor_postal_code,Safe predictor,Candidate predictor,Geographic information should be available before the prediction period and does not directly reveal the target.,"Evaluate cardinality, privacy, and fairness before modeling."
donor_age,Safe predictor,Candidate predictor,Demographic information does not directly reveal whether the individual donated during the target period.,Document that the exact age reference date is unavailable.
gender_identity,Safe predictor,Candidate predictor,Demographic information does not directly reveal the current-fiscal-year donation outcome.,Consolidate unknown and missing categories before modeling.
is_member_flag,Excluded feature,Exclude from predictors,The field is constant in the cleaned dataset and therefore provides no predictive variation.,Exclude from the modeling feature set.
is_alumnus_flag,Safe predictor,Candidate predictor,Alumni status is a relatively stable relationship attribute and does not directly reveal the target.,Retain unless later documentation identifies a timing issue.
is_parent_flag,Safe predictor,Candidate predictor,Parent status is a relatively stable relationship attribute and does not directly reveal the target.,Retain unless later documentation identifies a timing issue.
has_involvement_flag,Potential leakage,Exclude pending verification,"The snapshot date is unknown, so the field may include involvement established during the target period.",Confirm that the value was available at the prediction cutoff.
preferred_address_type,Safe predictor,Candidate predictor,Contact-preference information does not directly reveal current-fiscal-year donation behavior.,Standardize categories and handle missing values before modeling.
has_email_flag,Potential leakage,Exclude pending verification,The field may reflect contact information added or updated during the target period.,Confirm that the value was available at the prediction cutoff.


In [12]:
# Confirm that each column appears exactly once
duplicate_audit_columns = leakage_audit["column"].duplicated().sum()

dataset_columns = set(df.columns)
audited_columns = set(leakage_audit["column"])

missing_audit_columns = sorted(dataset_columns - audited_columns)
unexpected_audit_columns = sorted(audited_columns - dataset_columns)

invalid_classifications = sorted(
    set(leakage_audit["classification"])
    - VALID_LEAKAGE_CLASSIFICATIONS
)

assert duplicate_audit_columns == 0, (
    f"The leakage audit contains "
    f"{duplicate_audit_columns} duplicate column entries."
)

assert not missing_audit_columns, (
    f"Columns missing from the leakage audit: "
    f"{missing_audit_columns}"
)

assert not unexpected_audit_columns, (
    f"Audit columns not found in the dataset: "
    f"{unexpected_audit_columns}"
)

assert not invalid_classifications, (
    f"Invalid classifications found: "
    f"{invalid_classifications}"
)

print("Leakage audit validation passed.")
print(f"Dataset columns audited: {len(leakage_audit)}")

Leakage audit validation passed.
Dataset columns audited: 20


In [13]:
# Summarize the num of columns in each classification
leakage_audit_summary = (
    leakage_audit["classification"]
    .value_counts()
    .rename_axis("classification")
    .reset_index(name="column_count")
)

display(leakage_audit_summary.style.hide(axis="index"))

classification,column_count
Safe predictor,11
Potential leakage,4
Excluded feature,3
Identifier,1
Target,1


In [14]:
# Reusable column lists for feature engineering and modeling
SAFE_PREDICTOR_COLUMNS = leakage_audit.loc[
    leakage_audit["classification"] == "Safe predictor",
    "column",
].tolist()

POTENTIAL_LEAKAGE_COLUMNS = leakage_audit.loc[
    leakage_audit["classification"] == "Potential leakage",
    "column",
].tolist()

IDENTIFIER_COLUMNS = leakage_audit.loc[
    leakage_audit["classification"] == "Identifier",
    "column",
].tolist()

EXCLUDED_FEATURE_COLUMNS = leakage_audit.loc[
    leakage_audit["classification"] == "Excluded feature",
    "column",
].tolist()

TARGET_COLUMNS = leakage_audit.loc[
    leakage_audit["classification"] == "Target",
    "column",
].tolist()

print("Safe predictors:")
print(SAFE_PREDICTOR_COLUMNS)

print("\nPotential leakage columns:")
print(POTENTIAL_LEAKAGE_COLUMNS)

print("\nIdentifier columns:")
print(IDENTIFIER_COLUMNS)

print("\nExcluded features:")
print(EXCLUDED_FEATURE_COLUMNS)

print("\nTarget-related columns:")
print(TARGET_COLUMNS)

Safe predictors:
['donor_postal_code', 'donor_age', 'gender_identity', 'is_alumnus_flag', 'is_parent_flag', 'preferred_address_type', 'last_fiscal_year_donation', 'donation_2_fiscal_years_ago', 'donation_3_fiscal_years_ago', 'donation_4_fiscal_years_ago', 'donation_5_fiscal_years_ago']

Potential leakage columns:
['has_involvement_flag', 'has_email_flag', 'consecutive_donor_years', 'donor_indicator_flag']

Identifier columns:
['donor_unique_id']

Excluded features:
['is_member_flag', 'current_fiscal_year_donation', 'cumulative_donation_amount']

Target-related columns:
['target_current_fiscal_year_donor_flag']


## Feature Leakage Assessment

A formal leakage audit was completed for all 20 columns currently in the working dataset, including the newly created target variable.

Each column was assigned one of five classifications:

- Safe predictor: Available before the prediction period and does not directly reveal the target
- Potential leakage: May be usable, but its calculation timing or snapshot date must be verified
- Identifier: Used only for record tracking and not as a predictor
- Target: The binary outcome the model will predict
- Excluded feature: Must not be included in the predictor set because it is uninformative or directly risks leakage

The audit identified 11 safe predictors, consisting primarily of demographic, contact-preference, geographic, and completed historical donation variables.

The following four columns were classified as potential leakage:

- `has_involvement_flag`
- `has_email_flag`
- `consecutive_donor_years`
- `donor_indicator_flag`

`has_involvement_flag` and `has_email_flag` may be useful predictors, but their snapshot dates are unknown. They should remain excluded from the initial modeling feature set until it is confirmed that their values were available before the prediction cutoff.

`consecutive_donor_years` may include donation behavior from the current fiscal year. It must either be verified as a historical-only measure or replaced with a streak feature reconstructed from the five completed historical donation years.

`donor_indicator_flag` remains a historical donor-status field rather than the modeling target. However, its calculation logic and reference date are not currently known, so it will not be used as a predictor unless its timing is confirmed.

The following three columns were classified as excluded features:

- `is_member_flag`
- `current_fiscal_year_donation`
- `cumulative_donation_amount`

`is_member_flag` is excluded because it is constant throughout the finalized dataset and provides no predictive variation.

`current_fiscal_year_donation` directly determines the binary target and would cause target leakage if included as a predictor. It will be retained temporarily only for target construction and validation.

`cumulative_donation_amount` may contain current-fiscal-year donations and therefore may include information from the outcome period. The original column will remain excluded. A separate historical-value feature will be created using only donation information available before the prediction cutoff.

`donor_unique_id` was classified as an identifier. It will be retained for tracking records and connecting predictions back to individual donors, but it will not be included in the predictor matrix.

The final modeling target is `target_current_fiscal_year_donor_flag`.

The leakage audit provides the initial rules for predictor eligibility. Columns classified as potential leakage may be reconsidered if reliable documentation confirms that they were available before the prediction baseline.


In [15]:
# Define feature availability timeline
ESTIMATED_DEMOGRAPHIC_REFERENCE_PERIOD = (
    "Approximately 2018, based on the consistent eight-year difference "
    "between donor_age and age derived from donor_date_of_birth"
)

PREDICTION_BASELINE = (
    "Beginning of the dataset's current fiscal year "
    "(exact fiscal year and calendar date unconfirmed)"
)

HISTORICAL_OBSERVATION_PERIOD = (
    "Five completed fiscal years preceding the prediction baseline"
)

TARGET_OUTCOME_PERIOD = (
    "The dataset's current fiscal year"
)

print("Estimated demographic reference period:")
print(f"  {ESTIMATED_DEMOGRAPHIC_REFERENCE_PERIOD}")

print("\nPrediction baseline:")
print(f"  {PREDICTION_BASELINE}")

print("\nHistorical observation period:")
print(f"  {HISTORICAL_OBSERVATION_PERIOD}")

print("\nTarget outcome period:")
print(f"  {TARGET_OUTCOME_PERIOD}")

Estimated demographic reference period:
  Approximately 2018, based on the consistent eight-year difference between donor_age and age derived from donor_date_of_birth

Prediction baseline:
  Beginning of the dataset's current fiscal year (exact fiscal year and calendar date unconfirmed)

Historical observation period:
  Five completed fiscal years preceding the prediction baseline

Target outcome period:
  The dataset's current fiscal year


In [16]:
# Document feature availability relative to prediction cutoff
feature_availability = pd.DataFrame(
    [
        {
            "column_or_group": "Historical five-year donation columns",
            "availability_relative_to_cutoff": "Before cutoff",
            "feature_status": "Available",
            "notes": (
                "Represent five completed fiscal years before the "
                "target outcome period."
            ),
        },
        {
            "column_or_group": "current_fiscal_year_donation",
            "availability_relative_to_cutoff": "During outcome period",
            "feature_status": "Unavailable as predictor",
            "notes": (
                "Used only to construct and validate the binary target."
            ),
        },
        {
            "column_or_group": "cumulative_donation_amount",
            "availability_relative_to_cutoff": "Unknown",
            "feature_status": "Excluded",
            "notes": (
                "May include current-fiscal-year donations and must not "
                "be used directly."
            ),
        },
        {
            "column_or_group": "consecutive_donor_years",
            "availability_relative_to_cutoff": "Unknown",
            "feature_status": "Requires verification",
            "notes": (
                "May include donation activity from the current fiscal year."
            ),
        },
        {
            "column_or_group": "donor_indicator_flag",
            "availability_relative_to_cutoff": "Unknown",
            "feature_status": "Requires verification",
            "notes": (
                "Calculation logic and reference date are not confirmed."
            ),
        },
        {
            "column_or_group": "Demographic variables",
            "availability_relative_to_cutoff": "Provisionally before cutoff",
            "feature_status": "Provisionally available",
            "notes": (
                "donor_age appears to reflect an approximate 2018 reference "
                "period, but exact field snapshot dates are unconfirmed."
            ),
        },
        {
            "column_or_group": "Stable relationship variables",
            "availability_relative_to_cutoff": "Provisionally before cutoff",
            "feature_status": "Provisionally available",
            "notes": (
                "Includes is_alumnus_flag, is_parent_flag, and "
                "preferred_address_type."
            ),
        },
        {
            "column_or_group": "Engagement and contact variables",
            "availability_relative_to_cutoff": "Unknown",
            "feature_status": "Requires verification",
            "notes": (
                "Snapshot dates for has_involvement_flag and "
                "has_email_flag are not confirmed."
            ),
        },
        {
            "column_or_group": "donor_unique_id",
            "availability_relative_to_cutoff": "Not applicable",
            "feature_status": "Tracking only",
            "notes": (
                "Retained for record identification but excluded from predictors."
            ),
        },
        {
            "column_or_group": "target_current_fiscal_year_donor_flag",
            "availability_relative_to_cutoff": "During outcome period",
            "feature_status": "Target",
            "notes": (
                "Binary modeling outcome derived from "
                "current_fiscal_year_donation."
            ),
        },
    ]
)

display(feature_availability.style.hide(axis="index"))

column_or_group,availability_relative_to_cutoff,feature_status,notes
Historical five-year donation columns,Before cutoff,Available,Represent five completed fiscal years before the target outcome period.
current_fiscal_year_donation,During outcome period,Unavailable as predictor,Used only to construct and validate the binary target.
cumulative_donation_amount,Unknown,Excluded,May include current-fiscal-year donations and must not be used directly.
consecutive_donor_years,Unknown,Requires verification,May include donation activity from the current fiscal year.
donor_indicator_flag,Unknown,Requires verification,Calculation logic and reference date are not confirmed.
Demographic variables,Provisionally before cutoff,Provisionally available,"donor_age appears to reflect an approximate 2018 reference period, but exact field snapshot dates are unconfirmed."
Stable relationship variables,Provisionally before cutoff,Provisionally available,"Includes is_alumnus_flag, is_parent_flag, and preferred_address_type."
Engagement and contact variables,Unknown,Requires verification,Snapshot dates for has_involvement_flag and has_email_flag are not confirmed.
donor_unique_id,Not applicable,Tracking only,Retained for record identification but excluded from predictors.
target_current_fiscal_year_donor_flag,During outcome period,Target,Binary modeling outcome derived from current_fiscal_year_donation.


## Feature Availability Cutoff

To prevent temporal data leakage, every engineered predictor must be based only on information that would have been available when the prediction was made.

The feature engineering process uses the following timeline:

* Estimated demographic reference period: Approximately 2018, based on the consistent difference of eight years between `donor_age` and age derived from `donor_date_of_birth`
* Prediction baseline: The beginning of the dataset’s current fiscal year
* Historical observation period: The five completed fiscal years preceding the prediction baseline
* Target outcome period: The dataset’s current fiscal year

The exact fiscal year and calendar dates represented by the donation fields are not available. Therefore, the prediction cutoff is defined relative to the fiscal year labels in the dataset rather than an assumed calendar date.

During Phase 1, `donor_age` was consistently eight years lower than the age calculated from `donor_date_of_birth` across all valid records. Assuming the derived ages used a 2026 reference date, this suggests that `donor_age` reflects an approximate 2018 demographic reference period.

This finding indicates that at least some fields represent a historical snapshot. However, it does not confirm that the dataset’s current fiscal year is 2018 because demographic, engagement, contact, and donation fields may have been updated on different schedules.

The five historical donation columns are considered available before the prediction cutoff because they represent completed fiscal years preceding the target outcome period.

Based on the feature availability assessment:

* `current_fiscal_year_donation` belongs to the outcome period. It is used only to construct and validate `target_current_fiscal_year_donor_flag` and must not be used as a predictor.
* `cumulative_donation_amount` remains excluded because it may include donations from the current fiscal year. A separate historical value feature will be created using only donation information available before the prediction cutoff.
* `consecutive_donor_years` and `donor_indicator_flag` require additional verification because their calculation periods are unknown and may include activity from the current fiscal year.
* Demographic, geographic, contact preference, and stable relationship fields are provisionally considered available before the cutoff. `has_involvement_flag` and `has_email_flag` will remain unavailable until their snapshot timing is confirmed.
* `donor_unique_id` is retained only for record tracking, while `is_member_flag` remains excluded because it is constant in the finalized dataset.

All subsequent engineered features will be calculated using only information considered available before the prediction baseline.


In [17]:
# Define completed historical donation periods
HISTORICAL_DONATION_COLUMNS = [
    "last_fiscal_year_donation",
    "donation_2_fiscal_years_ago",
    "donation_3_fiscal_years_ago",
    "donation_4_fiscal_years_ago",
    "donation_5_fiscal_years_ago",
]

FEATURE_HISTORICAL_VALUE_COLUMN = (
    "feature_past_5yr_total_donation"
)

# Confirm that all required source columns are available
missing_historical_columns = sorted(
    set(HISTORICAL_DONATION_COLUMNS) - set(df.columns)
)

assert not missing_historical_columns, (
    "Missing historical donation columns: "
    f"{missing_historical_columns}"
)

# Validate the historical donation values
historical_missing_values = (
    df[HISTORICAL_DONATION_COLUMNS]
    .isna()
    .sum()
    .sum()
)

historical_negative_values = (
    df[HISTORICAL_DONATION_COLUMNS] < 0
).sum().sum()

print(
    "Missing historical donation values: "
    f"{historical_missing_values:,}"
)

print(
    "Negative historical donation values: "
    f"{historical_negative_values:,}"
)

assert historical_missing_values == 0, (
    "Historical donation columns contain missing values."
)

assert historical_negative_values == 0, (
    "Historical donation columns contain negative values."
)

Missing historical donation values: 0
Negative historical donation values: 0


In [18]:
# Create historical donation value feature
df[FEATURE_HISTORICAL_VALUE_COLUMN] = (
    df[HISTORICAL_DONATION_COLUMNS]
    .sum(axis=1)
)

print(
    "Historical value feature created: "
    f"{FEATURE_HISTORICAL_VALUE_COLUMN}"
)

Historical value feature created: feature_past_5yr_total_donation


In [19]:
# Validate historical value feature
expected_historical_5yr_total = (
    df[HISTORICAL_DONATION_COLUMNS]
    .sum(axis=1)
)

historical_value_mismatch_count = (
    ~np.isclose(
        df[FEATURE_HISTORICAL_VALUE_COLUMN],
        expected_historical_5yr_total,
    )
).sum()

assert historical_value_mismatch_count == 0, (
    "Found "
    f"{historical_value_mismatch_count:,} "
    "historical value calculation mismatches."
)

assert (
    df[FEATURE_HISTORICAL_VALUE_COLUMN] >= 0
).all(), (
    f"{FEATURE_HISTORICAL_VALUE_COLUMN} "
    "contains negative values."
)

print("Historical value feature validation passed.")

Historical value feature validation passed.


In [20]:
# Summarize historical value feature
historical_value_summary = pd.DataFrame(
    {
        "metric": [
            "Included donation periods",
            "Records",
            "Records with historical donations",
            "Records without historical donations",
            "Mean historical value",
            "Median historical value",
            "Maximum historical value",
        ],
        "value": [
            len(HISTORICAL_DONATION_COLUMNS),
            len(df),
            (
                df[FEATURE_HISTORICAL_VALUE_COLUMN] > 0
            ).sum(),
            (
                df[FEATURE_HISTORICAL_VALUE_COLUMN] == 0
            ).sum(),
            df[FEATURE_HISTORICAL_VALUE_COLUMN].mean(),
            df[FEATURE_HISTORICAL_VALUE_COLUMN].median(),
            df[FEATURE_HISTORICAL_VALUE_COLUMN].max(),
        ],
    }
)

display(historical_value_summary.style.format(
        {
            "value": lambda x: f"{int(x)}" if x % 1 == 0 else f"{x:.2f}"
        }
    ).hide(axis="index"))

metric,value
Included donation periods,5
Records,34403
Records with historical donations,9087
Records without historical donations,25316
Mean historical value,723.44
Median historical value,0
Maximum historical value,10200274


## Historical Donation Value

Based on the feature eligibility assessment, `cumulative_donation_amount` was classified as an excluded feature.

The original `cumulative_donation_amount` column may include donations from the current fiscal year. However, the dataset does not provide transaction dates or a verified fiscal year breakdown for the full cumulative amount. Subtracting `current_fiscal_year_donation` would require assuming that both columns were calculated using the same timing and accounting rules. Because that assumption has not been confirmed, `cumulative_donation_amount` will remain excluded from the predictor set.

Instead, the safer feature `feature_past_5yr_total_donation` was created by summing the five completed historical donation periods:

- `last_fiscal_year_donation`
- `donation_2_fiscal_years_ago`
- `donation_3_fiscal_years_ago`
- `donation_4_fiscal_years_ago`
- `donation_5_fiscal_years_ago`

The current fiscal year donation amount was not included because it belongs to the target outcome period.

`feature_past_5yr_total_donation` represents total giving during the available five year historical observation window. It does not represent the donor’s complete lifetime giving and should not be interpreted as lifetime value.


In [21]:
# Define the basic historical donation feature names
BASIC_HISTORICAL_FEATURE_COLUMNS = [
    "feature_past_5yr_total_donation",
    "feature_past_5yr_average_donation",
    "feature_past_5yr_median_donation",
    "feature_past_5yr_max_donation",
    "feature_past_5yr_min_donation",
    "feature_past_5yr_donation_std",
    "feature_past_5yr_active_average_donation",
]

assert "feature_past_5yr_total_donation" in df.columns, (
    "feature_past_5yr_total_donation was not found. "
    "Restart Kernal and Run all Cells."
)

print("Historical donation source columns:")
print(HISTORICAL_DONATION_COLUMNS)

print("\nBasic historical feature columns:")
print(BASIC_HISTORICAL_FEATURE_COLUMNS)

Historical donation source columns:
['last_fiscal_year_donation', 'donation_2_fiscal_years_ago', 'donation_3_fiscal_years_ago', 'donation_4_fiscal_years_ago', 'donation_5_fiscal_years_ago']

Basic historical feature columns:
['feature_past_5yr_total_donation', 'feature_past_5yr_average_donation', 'feature_past_5yr_median_donation', 'feature_past_5yr_max_donation', 'feature_past_5yr_min_donation', 'feature_past_5yr_donation_std', 'feature_past_5yr_active_average_donation']


In [22]:
# Summary features across the five historical donation periods
df["feature_past_5yr_average_donation"] = (
    df[HISTORICAL_DONATION_COLUMNS]
    .mean(axis=1)
)

df["feature_past_5yr_median_donation"] = (
    df[HISTORICAL_DONATION_COLUMNS]
    .median(axis=1)
)

df["feature_past_5yr_max_donation"] = (
    df[HISTORICAL_DONATION_COLUMNS]
    .max(axis=1)
)

df["feature_past_5yr_min_donation"] = (
    df[HISTORICAL_DONATION_COLUMNS]
    .min(axis=1)
)

df["feature_past_5yr_donation_std"] = (
    df[HISTORICAL_DONATION_COLUMNS]
    .std(axis=1, ddof=0)
)

positive_historical_donations = (
    df[HISTORICAL_DONATION_COLUMNS]
    .where(df[HISTORICAL_DONATION_COLUMNS] > 0)
)

df["feature_past_5yr_active_average_donation"] = (
    positive_historical_donations
    .mean(axis=1)
    .fillna(0.0)
)

print("Basic historical donation features created.")

Basic historical donation features created.


In [23]:
# Check features for missing, infinite, or negative values
historical_feature_missing_values = (
    df[BASIC_HISTORICAL_FEATURE_COLUMNS]
    .isna()
    .sum()
    .sum()
)

historical_feature_infinite_values = np.isinf(
    df[BASIC_HISTORICAL_FEATURE_COLUMNS]
).sum().sum()

historical_feature_negative_values = (
    df[BASIC_HISTORICAL_FEATURE_COLUMNS] < 0
).sum().sum()

print(
    "Missing engineered historical values: "
    f"{historical_feature_missing_values:,}"
)

print(
    "Infinite engineered historical values: "
    f"{historical_feature_infinite_values:,}"
)

print(
    "Negative engineered historical values: "
    f"{historical_feature_negative_values:,}"
)

assert historical_feature_missing_values == 0, (
    "Historical donation features contain missing values."
)

assert historical_feature_infinite_values == 0, (
    "Historical donation features contain infinite values."
)

assert historical_feature_negative_values == 0, (
    "Historical donation features contain negative values."
)

# Confirm that exactly five historical periods are being used
historical_period_count = len(HISTORICAL_DONATION_COLUMNS)

assert historical_period_count == 5, (
    "Past five year features require exactly five "
    "historical donation columns."
)

# Confirm that the average = the total ÷ the number of historical periods
average_mismatch_count = (
    ~np.isclose(
        df["feature_past_5yr_average_donation"],
        (
            df["feature_past_5yr_total_donation"]
            / historical_period_count
        ),
    )
).sum()

assert average_mismatch_count == 0, (
    f"Found {average_mismatch_count:,} "
    "average calculation mismatches."
)

inactive_average_mismatch_count = (
    (df["feature_past_5yr_total_donation"] == 0)
    & (
        ~np.isclose(
            df["feature_past_5yr_active_average_donation"],
            0.0,
        )
    )
).sum()

assert inactive_average_mismatch_count == 0, (
    "Records without historical donations have unexpected "
    "active average values."
)

# Confirm that excluding inactive years does not lower the average
active_average_order_mismatch_count = (
    df["feature_past_5yr_active_average_donation"]
    + 1e-10
    < df["feature_past_5yr_average_donation"]
).sum()

assert active_average_order_mismatch_count == 0, (
    "Active donation averages were unexpectedly lower than "
    "overall historical averages."
)

print("Basic historical donation feature validation passed.")

Missing engineered historical values: 0
Infinite engineered historical values: 0
Negative engineered historical values: 0
Basic historical donation feature validation passed.


In [24]:
# Summarize the basic historical donation features
basic_historical_feature_summary = (
    df[BASIC_HISTORICAL_FEATURE_COLUMNS]
    .describe()
    .transpose()
)

basic_historical_feature_summary["nonzero_records"] = (
    df[BASIC_HISTORICAL_FEATURE_COLUMNS] > 0
).sum()

basic_historical_feature_summary.index.name = "feature"

display(
    basic_historical_feature_summary.style.format(
        lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x
    )
)

,count,mean,std,min,25%,50%,75%,max,nonzero_records
feature,,,,,,,,,
feature_past_5yr_total_donation,"34,403",723.44,"58,342.93",0,0,0,1,"10,200,274","9,087"
feature_past_5yr_average_donation,"34,403",144.69,"11,668.59",0,0,0,0.2,"2,040,054.8","9,087"
feature_past_5yr_median_donation,"34,403",17.27,727.13,0,0,0,0,"90,000",407
feature_past_5yr_max_donation,"34,403",596.12,"55,992.71",0,0,0,1,"10,000,000","9,087"
feature_past_5yr_min_donation,"34,403",1.6,148.9,0,0,0,0,"25,000",10
feature_past_5yr_donation_std,"34,403",236.14,"22,261.85",0,0,0,0.4,"3,980,724.03","9,087"
feature_past_5yr_active_average_donation,"34,403",241.29,"15,401.43",0,0,0,1,"2,550,068.5","9,087"


## Historical Donation Summary Features

Basic donation value features were created using only the five completed fiscal years before the prediction period.

The previously created `feature_past_5yr_total_donation` feature was retained and used alongside six additional features:

- `feature_past_5yr_average_donation`
- `feature_past_5yr_median_donation`
- `feature_past_5yr_max_donation`
- `feature_past_5yr_min_donation`
- `feature_past_5yr_donation_std`
- `feature_past_5yr_active_average_donation`

The engineered features are:

- The overall average represents the average donation amount across all five historical years, including years with no donation.
- The median, maximum, and minimum summarize annual donation amounts within the historical observation window. Because zero represents a year without a donation, the median and minimum are zero for most individuals. The minimum is greater than zero only when an individual donated during all five historical years.
- The standard deviation measures how much annual donation amounts varied across the five year period. A population standard deviation was used because the calculation covers the complete historical observation window available in the dataset.
- The active average was calculated using only years in which the individual donated more than zero. It represents the typical amount donated during active giving years rather than averaging across inactive years. Individuals with no historical donations were assigned an active average of zero.

The feature distributions remain highly right skewed because a small number of donors contributed exceptionally large amounts. Skew resistant transformations will be created in a later feature engineering step.

No current fiscal year donation values or cumulative donation values were used to create these features.


In [25]:
# Define historical donation frequency feature
DONATION_FREQUENCY_FEATURE_COLUMNS = [
    "feature_years_donated_past_5yr",
    "feature_past_5yr_donation_frequency_rate",
    "feature_any_past_donation_flag",
    "feature_multiple_year_donor_flag",
    "feature_consistent_donor_flag",
]

historical_period_count = len(HISTORICAL_DONATION_COLUMNS)

assert historical_period_count == 5, (
    "Past five year frequency features require exactly five "
    "historical donation columns."
)

print("Donation frequency feature columns:")
print(DONATION_FREQUENCY_FEATURE_COLUMNS)

Donation frequency feature columns:
['feature_years_donated_past_5yr', 'feature_past_5yr_donation_frequency_rate', 'feature_any_past_donation_flag', 'feature_multiple_year_donor_flag', 'feature_consistent_donor_flag']


In [26]:
# Donation freq. features
historical_donation_activity = (
    df[HISTORICAL_DONATION_COLUMNS] > 0
)

df["feature_years_donated_past_5yr"] = (
    historical_donation_activity
    .sum(axis=1)
    .astype("int8")
)

df["feature_past_5yr_donation_frequency_rate"] = (
    df["feature_years_donated_past_5yr"]
    / historical_period_count
)

df["feature_any_past_donation_flag"] = (
    df["feature_years_donated_past_5yr"] >= 1
).astype("int8")

df["feature_multiple_year_donor_flag"] = (
    df["feature_years_donated_past_5yr"] >= 2
).astype("int8")

df["feature_consistent_donor_flag"] = (
    df["feature_years_donated_past_5yr"] >= 3
).astype("int8")

print("Donation frequency features created.")

Donation frequency features created.


In [27]:
# Validate donation freq. features
frequency_feature_missing_values = (
    df[DONATION_FREQUENCY_FEATURE_COLUMNS]
    .isna()
    .sum()
    .sum()
)

frequency_feature_infinite_values = np.isinf(
    df[DONATION_FREQUENCY_FEATURE_COLUMNS]
).sum().sum()

frequency_feature_negative_values = (
    df[DONATION_FREQUENCY_FEATURE_COLUMNS] < 0
).sum().sum()

print(
    "Missing donation frequency values: "
    f"{frequency_feature_missing_values:,}"
)

print(
    "Infinite donation frequency values: "
    f"{frequency_feature_infinite_values:,}"
)

print(
    "Negative donation frequency values: "
    f"{frequency_feature_negative_values:,}"
)

assert frequency_feature_missing_values == 0, (
    "Donation frequency features contain missing values."
)

assert frequency_feature_infinite_values == 0, (
    "Donation frequency features contain infinite values."
)

assert frequency_feature_negative_values == 0, (
    "Donation frequency features contain negative values."
)

# Confirm num of active years is in between 0 and 5
assert df["feature_years_donated_past_5yr"].between(
    0,
    historical_period_count,
).all(), (
    "Historical donation year counts fall outside the expected range."
)

# Confirm frequency rate falls between 0 and 1
assert df["feature_past_5yr_donation_frequency_rate"].between(
    0,
    1,
).all(), (
    "Donation frequency rates fall outside the expected range."
)

# Confirm frequency rate matches the active year count
frequency_rate_mismatch_count = (
    ~np.isclose(
        df["feature_past_5yr_donation_frequency_rate"],
        (
            df["feature_years_donated_past_5yr"]
            / historical_period_count
        ),
    )
).sum()

assert frequency_rate_mismatch_count == 0, (
    f"Found {frequency_rate_mismatch_count:,} "
    "donation frequency rate mismatches."
)

any_donation_mismatch_count = (
    df["feature_any_past_donation_flag"]
    != (
        df["feature_years_donated_past_5yr"] >= 1
    ).astype("int8")
).sum()

multiple_year_mismatch_count = (
    df["feature_multiple_year_donor_flag"]
    != (
        df["feature_years_donated_past_5yr"] >= 2
    ).astype("int8")
).sum()

consistent_donor_mismatch_count = (
    df["feature_consistent_donor_flag"]
    != (
        df["feature_years_donated_past_5yr"] >= 3
    ).astype("int8")
).sum()

assert any_donation_mismatch_count == 0, (
    f"Found {any_donation_mismatch_count:,} "
    "any donation flag mismatches."
)

assert multiple_year_mismatch_count == 0, (
    f"Found {multiple_year_mismatch_count:,} "
    "multiple year donor flag mismatches."
)

assert consistent_donor_mismatch_count == 0, (
    f"Found {consistent_donor_mismatch_count:,} "
    "consistent donor flag mismatches."
)

frequency_hierarchy_mismatch_count = (
    (
        df["feature_consistent_donor_flag"]
        > df["feature_multiple_year_donor_flag"]
    )
    | (
        df["feature_multiple_year_donor_flag"]
        > df["feature_any_past_donation_flag"]
    )
).sum()

assert frequency_hierarchy_mismatch_count == 0, (
    "Donation frequency flags do not follow the expected hierarchy."
)

print("Donation frequency feature validation passed.")

Missing donation frequency values: 0
Infinite donation frequency values: 0
Negative donation frequency values: 0
Donation frequency feature validation passed.


In [28]:
# Summarize distribution of active donation years
donation_frequency_distribution = (
    df["feature_years_donated_past_5yr"]
    .value_counts()
    .sort_index()
    .rename_axis("years_donated_past_5yr")
    .reset_index(name="record_count")
)

donation_frequency_distribution["percentage"] = (
    donation_frequency_distribution["record_count"]
    / len(df)
    * 100
).round(2)

display(
    donation_frequency_distribution.style.format({
        "percentage": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') + "%" if isinstance(x, (int, float)) else x,
        "record_count": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x
    }).hide(axis="index")
)

years_donated_past_5yr,record_count,percentage
0,"25,316",73.59%
1,"7,193",20.91%
2,"1,487",4.32%
3,347,1.01%
4,50,0.15%
5,10,0.03%


In [29]:
# Summarize the binary donation frequency indicators
donation_frequency_flag_summary = pd.DataFrame(
    {
        "feature": [
            "feature_any_past_donation_flag",
            "feature_multiple_year_donor_flag",
            "feature_consistent_donor_flag",
        ],
        "records_flagged": [
            df["feature_any_past_donation_flag"].sum(),
            df["feature_multiple_year_donor_flag"].sum(),
            df["feature_consistent_donor_flag"].sum(),
        ],
    }
)

donation_frequency_flag_summary["percentage"] = (
    donation_frequency_flag_summary["records_flagged"]
    / len(df)
    * 100
).round(2)

display(
    donation_frequency_flag_summary.style.format({
        "percentage": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') + "%" if isinstance(x, (int, float)) else x,
        "records_flagged": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x
    }).hide(axis="index")
)

feature,records_flagged,percentage
feature_any_past_donation_flag,"9,087",26.41%
feature_multiple_year_donor_flag,"1,894",5.51%
feature_consistent_donor_flag,407,1.18%


## Historical Donation Frequency

Donation frequency features were created using the five completed fiscal years before the target outcome period.

The following features were created:
- `feature_years_donated_past_5yr` counts the number of historical years in which an individual donated more than zero.
- `feature_past_5yr_donation_frequency_rate` divides the number of active donation years by five. The feature ranges from 0 to 1, where 0 represents no historical donations and 1 represents donations during all five historical years.
- `feature_any_past_donation_flag` identifies individuals who donated during at least one historical year.
- `feature_multiple_year_donor_flag` identifies individuals who donated during at least two historical years.
- `feature_consistent_donor_flag` identifies individuals who donated during at least three historical years.

The binary thresholds provide simple donor frequency groups, but they are not treated as final assumptions about donor behavior. These features measure how often an individual donated, but they do not describe whether the donations occurred consecutively.

No current fiscal year donation information was used to create these features.


In [30]:
# Define streak features
ACTIVE_YEAR_COUNT_COLUMN = "feature_years_donated_past_5yr"
FREQUENCY_RATE_COLUMN = "feature_past_5yr_donation_frequency_rate"
REPEAT_DONOR_COLUMN = "feature_multiple_year_donor_flag"

MAX_STREAK_COLUMN = "feature_max_donation_streak_past_5yr"
INTERMITTENT_DONOR_COLUMN = "feature_intermittent_donor_flag"

NEW_STREAK_FEATURE_COLUMNS = [
    MAX_STREAK_COLUMN,
    INTERMITTENT_DONOR_COLUMN,
]

required_frequency_columns = [
    ACTIVE_YEAR_COUNT_COLUMN,
    FREQUENCY_RATE_COLUMN,
    REPEAT_DONOR_COLUMN,
]

missing_frequency_columns = sorted(
    set(required_frequency_columns) - set(df.columns)
)

assert not missing_frequency_columns, (
    "Required donation frequency features are missing: "
    f"{missing_frequency_columns}"
)

print("Frequency features:")
print(required_frequency_columns)

print("\nStreak features:")
print(NEW_STREAK_FEATURE_COLUMNS)

Frequency features:
['feature_years_donated_past_5yr', 'feature_past_5yr_donation_frequency_rate', 'feature_multiple_year_donor_flag']

Streak features:
['feature_max_donation_streak_past_5yr', 'feature_intermittent_donor_flag']


In [31]:
# Oldest to recent historical donation columns
chronological_historical_columns = list(
    reversed(HISTORICAL_DONATION_COLUMNS)
)

# Convert historical donation into annual activity indicators
historical_activity_chronological = (
    df[chronological_historical_columns]
    .gt(0)
    .to_numpy(dtype="int8")
)

# Calculate longest consecutive donation streak for each record
current_streak = np.zeros(len(df), dtype="int8")
maximum_streak = np.zeros(len(df), dtype="int8")

for period_activity in historical_activity_chronological.T:
    current_streak = np.where(
        period_activity == 1,
        current_streak + 1,
        0,
    )

    maximum_streak = np.maximum(
        maximum_streak,
        current_streak,
    )

df[MAX_STREAK_COLUMN] = maximum_streak

# Repeat donors whose active years were separated by 1 inactive year
df[INTERMITTENT_DONOR_COLUMN] = (
    (
        df[ACTIVE_YEAR_COUNT_COLUMN] >= 2
    )
    & (
        df[MAX_STREAK_COLUMN]
        < df[ACTIVE_YEAR_COUNT_COLUMN]
    )
).astype("int8")

print("Historical donation streak features created.")

Historical donation streak features created.


In [32]:
# Check the new streak features for invalid values
streak_feature_missing_values = (
    df[NEW_STREAK_FEATURE_COLUMNS]
    .isna()
    .sum()
    .sum()
)

streak_feature_infinite_values = np.isinf(
    df[NEW_STREAK_FEATURE_COLUMNS]
).sum().sum()

streak_feature_negative_values = (
    df[NEW_STREAK_FEATURE_COLUMNS] < 0
).sum().sum()

print(
    "Missing donation streak values: "
    f"{streak_feature_missing_values:,}"
)

print(
    "Infinite donation streak values: "
    f"{streak_feature_infinite_values:,}"
)

print(
    "Negative donation streak values: "
    f"{streak_feature_negative_values:,}"
)

assert streak_feature_missing_values == 0, (
    "Donation streak features contain missing values."
)

assert streak_feature_infinite_values == 0, (
    "Donation streak features contain infinite values."
)

assert streak_feature_negative_values == 0, (
    "Donation streak features contain negative values."
)

assert df[MAX_STREAK_COLUMN].between(
    0,
    historical_period_count,
).all(), (
    "Historical donation streak values fall outside "
    "the expected range."
)

streak_exceeds_active_years_count = (
    df[MAX_STREAK_COLUMN]
    > df[ACTIVE_YEAR_COUNT_COLUMN]
).sum()

assert streak_exceeds_active_years_count == 0, (
    f"Found {streak_exceeds_active_years_count:,} records "
    "where the maximum streak exceeds the active year count."
)

inactive_streak_mismatch_count = (
    (df[ACTIVE_YEAR_COUNT_COLUMN] == 0)
    & (df[MAX_STREAK_COLUMN] != 0)
).sum()

assert inactive_streak_mismatch_count == 0, (
    "Records without historical donations have "
    "unexpected streak values."
)

active_streak_mismatch_count = (
    (df[ACTIVE_YEAR_COUNT_COLUMN] > 0)
    & (df[MAX_STREAK_COLUMN] < 1)
).sum()

assert active_streak_mismatch_count == 0, (
    "Records with historical donations have "
    "unexpected streak values."
)

repeat_donor_mismatch_count = (
    df[REPEAT_DONOR_COLUMN]
    != (
        df[ACTIVE_YEAR_COUNT_COLUMN] >= 2
    ).astype("int8")
).sum()

assert repeat_donor_mismatch_count == 0, (
    f"Found {repeat_donor_mismatch_count:,} "
    "repeat donor definition mismatches."
)

expected_intermittent_donor_flag = (
    (
        df[ACTIVE_YEAR_COUNT_COLUMN] >= 2
    )
    & (
        df[MAX_STREAK_COLUMN]
        < df[ACTIVE_YEAR_COUNT_COLUMN]
    )
).astype("int8")

intermittent_donor_mismatch_count = (
    df[INTERMITTENT_DONOR_COLUMN]
    != expected_intermittent_donor_flag
).sum()

assert intermittent_donor_mismatch_count == 0, (
    f"Found {intermittent_donor_mismatch_count:,} "
    "intermittent donor definition mismatches."
)

intermittent_hierarchy_mismatch_count = (
    df[INTERMITTENT_DONOR_COLUMN]
    > df[REPEAT_DONOR_COLUMN]
).sum()

assert intermittent_hierarchy_mismatch_count == 0, (
    "Intermittent donor flags do not follow "
    "the expected repeat donor hierarchy."
)

print("Historical donation streak feature validation passed.")

Missing donation streak values: 0
Infinite donation streak values: 0
Negative donation streak values: 0
Historical donation streak feature validation passed.


In [33]:
# Summarize the maximum historical donation streak
donation_streak_distribution = (
    df[MAX_STREAK_COLUMN]
    .value_counts()
    .sort_index()
    .rename_axis("maximum_donation_streak_past_5yr")
    .reset_index(name="record_count")
)

donation_streak_distribution["percentage"] = (
    donation_streak_distribution["record_count"]
    / len(df)
    * 100
).round(2)

display(
    donation_streak_distribution.style.format({
        "percentage": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') + "%" if isinstance(x, (int, float)) else x,
        "record_count": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x
    }).hide(axis="index")
)

maximum_donation_streak_past_5yr,record_count,percentage
0,"25,316",73.59%
1,"7,839",22.79%
2,991,2.88%
3,207,0.6%
4,40,0.12%
5,10,0.03%


In [34]:
# Summarize repeat and intermittent donor patterns
donor_pattern_summary = pd.DataFrame(
    {
        "donor_pattern": [
            "Repeat donors",
            "Intermittent donors",
        ],
        "record_count": [
            df[REPEAT_DONOR_COLUMN].sum(),
            df[INTERMITTENT_DONOR_COLUMN].sum(),
        ],
    }
)

donor_pattern_summary["percentage"] = (
    donor_pattern_summary["record_count"]
    / len(df)
    * 100
).round(2)

display(
    donor_pattern_summary.style.format({
        "percentage": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') + "%" if isinstance(x, (int, float)) else x,
        "record_count": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x
    }).hide(axis="index")
)

donor_pattern,record_count,percentage
Repeat donors,"1,894",5.51%
Intermittent donors,795,2.31%


In [35]:
# Compare num of active years with max streak
frequency_streak_comparison = pd.crosstab(
    df[ACTIVE_YEAR_COUNT_COLUMN],
    df[MAX_STREAK_COLUMN],
)

frequency_streak_comparison.index.name = (
    "years_donated_past_5yr"
)

frequency_streak_comparison.columns.name = (
    "maximum_donation_streak_past_5yr"
)

display(frequency_streak_comparison)

maximum_donation_streak_past_5yr,0,1,2,3,4,5
years_donated_past_5yr,,,,,,
0,25316,0,0,0,0,0
1,0,7193,0,0,0,0
2,0,637,850,0,0,0
3,0,9,139,199,0,0
4,0,0,2,8,40,0
5,0,0,0,0,0,10


## Historical Donation Streaks

Donation frequency and donation streaks describe different aspects of historical donor behavior. Frequency measures how many years an individual donated, while a streak measures whether those donations occurred in consecutive years.

The previously created frequency features were retained:
- `feature_years_donated_past_5yr`
- `feature_past_5yr_donation_frequency_rate`
- `feature_multiple_year_donor_flag`

`feature_multiple_year_donor_flag` already identifies repeat donors who donated during at least two historical years. A separate repeat donor feature was not created because it would duplicate the same information.

Two new streak features were created:

- `feature_max_donation_streak_past_5yr`: records the longest sequence of consecutive donation years within the five year historical window.
- `feature_intermittent_donor_flag`: identifies repeat donors whose active donation years were separated by at least one inactive year.

The results show that 1,894 individuals donated during multiple historical years. Of these, 795 had an intermittent donation pattern.

Frequency and streak features provide different information. Two individuals may have donated during the same number of historical years while having different streak patterns. For example, one may have donated during two consecutive years, while another may have donated during two separated years.

The original `consecutive_donor_years` field was not used because its calculation period has not been verified and may include activity from the current fiscal year.

No current fiscal year donation information was used to create these features.


In [36]:
# Define donation recency features
YEARS_SINCE_LAST_DONATION_COLUMN = (
    "feature_years_since_last_donation_past_5yr"
)

DONATED_LAST_YEAR_COLUMN = (
    "feature_donated_last_year_flag"
)

DONATED_WITHIN_2_YEARS_COLUMN = (
    "feature_donated_within_2_years_flag"
)

LAPSED_DONOR_COLUMN = (
    "feature_lapsed_donor_flag"
)

NEVER_DONATED_COLUMN = (
    "feature_never_donated_past_5yr_flag"
)

MOST_RECENT_POSITIVE_DONATION_COLUMN = (
    "feature_most_recent_positive_donation"
)

DONATION_RECENCY_FEATURE_COLUMNS = [
    YEARS_SINCE_LAST_DONATION_COLUMN,
    DONATED_LAST_YEAR_COLUMN,
    DONATED_WITHIN_2_YEARS_COLUMN,
    LAPSED_DONOR_COLUMN,
    NEVER_DONATED_COLUMN,
    MOST_RECENT_POSITIVE_DONATION_COLUMN,
]

historical_period_count = len(HISTORICAL_DONATION_COLUMNS)

assert historical_period_count == 5, (
    "Past five year recency features require exactly five "
    "historical donation columns."
)

NO_HISTORICAL_DONATION_RECENCY_VALUE = 6

print("Donation recency feature columns:")
print(DONATION_RECENCY_FEATURE_COLUMNS)

print(
    "\nNo historical donation recency value: "
    f"{NO_HISTORICAL_DONATION_RECENCY_VALUE}"
)

Donation recency feature columns:
['feature_years_since_last_donation_past_5yr', 'feature_donated_last_year_flag', 'feature_donated_within_2_years_flag', 'feature_lapsed_donor_flag', 'feature_never_donated_past_5yr_flag', 'feature_most_recent_positive_donation']

No historical donation recency value: 6


In [37]:
# Create recency features
historical_donation_amounts = (
    df[HISTORICAL_DONATION_COLUMNS]
)

historical_donation_activity = (
    historical_donation_amounts > 0
)

has_historical_donation = (
    historical_donation_activity.any(axis=1)
)

most_recent_donation_position = (
    historical_donation_activity
    .to_numpy()
    .argmax(axis=1)
)

df[YEARS_SINCE_LAST_DONATION_COLUMN] = np.where(
    has_historical_donation,
    most_recent_donation_position,
    NO_HISTORICAL_DONATION_RECENCY_VALUE,
).astype("int8")

df[DONATED_LAST_YEAR_COLUMN] = (
    historical_donation_activity
    .iloc[:, 0]
    .astype("int8")
)

df[DONATED_WITHIN_2_YEARS_COLUMN] = (
    historical_donation_activity
    .iloc[:, :2]
    .any(axis=1)
    .astype("int8")
)

df[LAPSED_DONOR_COLUMN] = (
    has_historical_donation
    & (
        df[DONATED_WITHIN_2_YEARS_COLUMN] == 0
    )
).astype("int8")

df[NEVER_DONATED_COLUMN] = (
    ~has_historical_donation
).astype("int8")

df[MOST_RECENT_POSITIVE_DONATION_COLUMN] = (
    historical_donation_amounts
    .where(historical_donation_amounts > 0)
    .bfill(axis=1)
    .iloc[:, 0]
    .fillna(0.0)
)

print("Donation recency features created.")

Donation recency features created.


In [38]:
# Validate recency features
recency_feature_missing_values = (
    df[DONATION_RECENCY_FEATURE_COLUMNS]
    .isna()
    .sum()
    .sum()
)

recency_feature_infinite_values = np.isinf(
    df[DONATION_RECENCY_FEATURE_COLUMNS]
).sum().sum()

recency_feature_negative_values = (
    df[DONATION_RECENCY_FEATURE_COLUMNS] < 0
).sum().sum()

print(
    "Missing donation recency values: "
    f"{recency_feature_missing_values:,}"
)

print(
    "Infinite donation recency values: "
    f"{recency_feature_infinite_values:,}"
)

print(
    "Negative donation recency values: "
    f"{recency_feature_negative_values:,}"
)

assert recency_feature_missing_values == 0, (
    "Donation recency features contain missing values."
)

assert recency_feature_infinite_values == 0, (
    "Donation recency features contain infinite values."
)

assert recency_feature_negative_values == 0, (
    "Donation recency features contain negative values."
)

expected_recency_values = {
    0,
    1,
    2,
    3,
    4,
    NO_HISTORICAL_DONATION_RECENCY_VALUE,
}

actual_recency_values = set(
    df[YEARS_SINCE_LAST_DONATION_COLUMN].unique()
)

assert actual_recency_values.issubset(
    expected_recency_values
), (
    "Unexpected donation recency values found: "
    f"{actual_recency_values}"
)

last_year_mismatch_count = (
    df[DONATED_LAST_YEAR_COLUMN]
    != (
        df[YEARS_SINCE_LAST_DONATION_COLUMN] == 0
    ).astype("int8")
).sum()

within_2_years_mismatch_count = (
    df[DONATED_WITHIN_2_YEARS_COLUMN]
    != (
        df[YEARS_SINCE_LAST_DONATION_COLUMN]
        .isin([0, 1])
    ).astype("int8")
).sum()

lapsed_donor_mismatch_count = (
    df[LAPSED_DONOR_COLUMN]
    != (
        df[YEARS_SINCE_LAST_DONATION_COLUMN]
        .isin([2, 3, 4])
    ).astype("int8")
).sum()

never_donated_mismatch_count = (
    df[NEVER_DONATED_COLUMN]
    != (
        df[YEARS_SINCE_LAST_DONATION_COLUMN]
        == NO_HISTORICAL_DONATION_RECENCY_VALUE
    ).astype("int8")
).sum()

assert last_year_mismatch_count == 0, (
    f"Found {last_year_mismatch_count:,} "
    "last year donation mismatches."
)

assert within_2_years_mismatch_count == 0, (
    f"Found {within_2_years_mismatch_count:,} "
    "two year donation mismatches."
)

assert lapsed_donor_mismatch_count == 0, (
    f"Found {lapsed_donor_mismatch_count:,} "
    "lapsed donor mismatches."
)

assert never_donated_mismatch_count == 0, (
    f"Found {never_donated_mismatch_count:,} "
    "never donated mismatches."
)

recent_flag_hierarchy_mismatch_count = (
    df[DONATED_LAST_YEAR_COLUMN]
    > df[DONATED_WITHIN_2_YEARS_COLUMN]
).sum()

assert recent_flag_hierarchy_mismatch_count == 0, (
    "Last year donor flags do not follow the expected hierarchy."
)

recency_partition_count = (
    df[DONATED_WITHIN_2_YEARS_COLUMN]
    + df[LAPSED_DONOR_COLUMN]
    + df[NEVER_DONATED_COLUMN]
)

assert (recency_partition_count == 1).all(), (
    "Recency groups do not form a complete and exclusive partition."
)


expected_most_recent_positive_donation = (
    historical_donation_amounts
    .where(historical_donation_amounts > 0)
    .bfill(axis=1)
    .iloc[:, 0]
    .fillna(0.0)
)

recent_amount_mismatch_count = (
    ~np.isclose(
        df[MOST_RECENT_POSITIVE_DONATION_COLUMN],
        expected_most_recent_positive_donation,
    )
).sum()

assert recent_amount_mismatch_count == 0, (
    f"Found {recent_amount_mismatch_count:,} "
    "most recent donation amount mismatches."
)

print("Donation recency feature validation passed.")

Missing donation recency values: 0
Infinite donation recency values: 0
Negative donation recency values: 0
Donation recency feature validation passed.


In [39]:
# Summarize years since the most recent historical donation
donation_recency_distribution = (
    df[YEARS_SINCE_LAST_DONATION_COLUMN]
    .value_counts()
    .sort_index()
    .rename_axis("years_since_last_donation")
    .reset_index(name="record_count")
)

donation_recency_distribution["recency_label"] = (
    donation_recency_distribution[
        "years_since_last_donation"
    ].map(
        {
            0: "Donated last fiscal year",
            1: "Last donated two fiscal years ago",
            2: "Last donated three fiscal years ago",
            3: "Last donated four fiscal years ago",
            4: "Last donated five fiscal years ago",
            6: "No donation in historical window",
        }
    )
)

donation_recency_distribution["percentage"] = (
    donation_recency_distribution["record_count"]
    / len(df)
    * 100
).round(2)

donation_recency_distribution = (
    donation_recency_distribution[
        [
            "years_since_last_donation",
            "recency_label",
            "record_count",
            "percentage",
        ]
    ]
)

display(
    donation_recency_distribution.style.format({
        "percentage": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') + "%" if isinstance(x, (int, float)) else x,
        "record_count": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x
    }).hide(axis="index")
)

years_since_last_donation,recency_label,record_count,percentage
0,Donated last fiscal year,"2,358",6.85%
1,Last donated two fiscal years ago,"2,240",6.51%
2,Last donated three fiscal years ago,"1,549",4.5%
3,Last donated four fiscal years ago,"1,485",4.32%
4,Last donated five fiscal years ago,"1,455",4.23%
6,No donation in historical window,"25,316",73.59%


In [40]:
# Summarize the recency pattern flags
donation_recency_flag_summary = pd.DataFrame(
    {
        "feature": [
            DONATED_LAST_YEAR_COLUMN,
            DONATED_WITHIN_2_YEARS_COLUMN,
            LAPSED_DONOR_COLUMN,
            NEVER_DONATED_COLUMN,
        ],
        "records_flagged": [
            df[DONATED_LAST_YEAR_COLUMN].sum(),
            df[DONATED_WITHIN_2_YEARS_COLUMN].sum(),
            df[LAPSED_DONOR_COLUMN].sum(),
            df[NEVER_DONATED_COLUMN].sum(),
        ],
    }
)

donation_recency_flag_summary["percentage"] = (
    donation_recency_flag_summary["records_flagged"]
    / len(df)
    * 100
).round(2)

display(
    donation_recency_flag_summary.style.format({
        "percentage": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') + "%" if isinstance(x, (int, float)) else x,
        "records_flagged": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x
    }).hide(axis="index")
)

feature,records_flagged,percentage
feature_donated_last_year_flag,"2,358",6.85%
feature_donated_within_2_years_flag,"4,598",13.37%
feature_lapsed_donor_flag,"4,489",13.05%
feature_never_donated_past_5yr_flag,"25,316",73.59%


In [41]:
# Summarize the most recent positive donation amount
most_recent_donation_summary = (
    df[[MOST_RECENT_POSITIVE_DONATION_COLUMN]]
    .describe()
    .transpose()
)

most_recent_donation_summary["nonzero_records"] = (
    df[MOST_RECENT_POSITIVE_DONATION_COLUMN] > 0
).sum()

most_recent_donation_summary.index.name = "feature"

display(
    most_recent_donation_summary.style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "center")]}
    ])
    .format(
        lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x
    )
)

,count,mean,std,min,25%,50%,75%,max,nonzero_records
feature,,,,,,,,,
feature_most_recent_positive_donation,"34,403",440.69,"54,180.69",0,0,0,1,"10,000,000","9,087"


## Historical Donation Recency

Donation recency features were created using the five completed fiscal years before the target outcome period.

The following features were created:
- `feature_years_since_last_donation_past_5yr` measures how many historical periods have passed since the most recent donation. A value of 0 represents a donation during the most recent completed fiscal year, while values from 1 through 4 represent progressively older donations.
- `feature_donated_last_year_flag` identifies individuals who donated during the most recent completed fiscal year. This feature also serves as the recent donor indicator, so a separate feature with the same definition was not created.
- `feature_donated_within_2_years_flag` identifies individuals who donated during either of the two most recent completed fiscal years.
- `feature_lapsed_donor_flag` identifies individuals who donated during the historical window but not during either of the two most recent fiscal years.
- `feature_never_donated_past_5yr_flag` identifies individuals with no donations during the five year historical window.
- `feature_most_recent_positive_donation` records the amount of the most recent positive donation found within the historical window.

Individuals with no historical donations were assigned a recency value of 6 rather than a missing value. This value represents no observed donation within the available historical window and should not be interpreted as exactly six years since the individual last donated.

The results show that 4,598 individuals donated within the two most recent historical years, 4,489 were classified as lapsed donors, and 25,316 had no observed donations during the five year historical window.

No current fiscal year donation information was used to create these features.


In [42]:
# Identify columns that may contain date or timestamp information
def infer_recency_resolution(
    dataframe,
    historical_donation_columns,
):
    """
    Infer the most precise recency resolution supported by
    the available dataset columns.
    """
    column_names = [
        column.lower()
        for column in dataframe.columns
    ]

    date_keywords = [
        "date",
        "datetime",
        "timestamp",
    ]

    month_keywords = [
        "month",
        "monthly",
    ]

    date_columns = [
        column
        for column in dataframe.columns
        if any(
            keyword in column.lower()
            for keyword in date_keywords
        )
    ]

    month_columns = [
        column
        for column in dataframe.columns
        if any(
            keyword in column.lower()
            for keyword in month_keywords
        )
    ]

    available_historical_columns = [
        column
        for column in historical_donation_columns
        if column in dataframe.columns
    ]

    fiscal_year_columns = [
        column
        for column in available_historical_columns
        if "fiscal_year" in column.lower()
    ]

    if date_columns:
        return {
            "resolution": "Days",
            "supporting_columns": date_columns,
            "reason": (
                "One or more date or timestamp columns are available."
            ),
        }

    if month_columns:
        return {
            "resolution": "Months",
            "supporting_columns": month_columns,
            "reason": (
                "Monthly fields are available, but exact donation "
                "dates are not present."
            ),
        }

    if (
        fiscal_year_columns
        and len(fiscal_year_columns)
        == len(historical_donation_columns)
    ):
        return {
            "resolution": "Fiscal years",
            "supporting_columns": fiscal_year_columns,
            "reason": (
                "Donation history is available only as annual "
                "fiscal year totals."
            ),
        }

    return {
        "resolution": "Unknown",
        "supporting_columns": [],
        "reason": (
            "The dataset does not contain enough temporal "
            "information to determine recency resolution."
        ),
    }


recency_resolution_result = infer_recency_resolution(
    dataframe=df,
    historical_donation_columns=HISTORICAL_DONATION_COLUMNS,
)

RECENCY_RESOLUTION = (
    recency_resolution_result["resolution"]
)

print(
    "Supported recency resolution: "
    f"{RECENCY_RESOLUTION}"
)

print(
    "Reason: "
    f"{recency_resolution_result['reason']}"
)

print("Supporting columns:")
print(
    recency_resolution_result["supporting_columns"]
)

Supported recency resolution: Fiscal years
Reason: Donation history is available only as annual fiscal year totals.
Supporting columns:
['last_fiscal_year_donation', 'donation_2_fiscal_years_ago', 'donation_3_fiscal_years_ago', 'donation_4_fiscal_years_ago', 'donation_5_fiscal_years_ago']


In [43]:
# Define recency values
RECENCY_FEATURE_COLUMN = (
    "feature_years_since_last_donation_past_5yr"
)

historical_period_count = len(
    HISTORICAL_DONATION_COLUMNS
)

NO_HISTORICAL_DONATION_RECENCY_VALUE = (
    historical_period_count + 1
)

assert RECENCY_FEATURE_COLUMN in df.columns, (
    f"{RECENCY_FEATURE_COLUMN} was not found. "
    "Complete the donation recency feature task "
    "before continuing."
)

assert RECENCY_RESOLUTION == "Fiscal years", (
    "The existing recency feature assumes fiscal year "
    "level donation history."
)

print(f"Recency feature: {RECENCY_FEATURE_COLUMN}")

print(
    "Historical periods available: "
    f"{historical_period_count}"
)

print(
    "No historical donation value: "
    f"{NO_HISTORICAL_DONATION_RECENCY_VALUE}"
)

Recency feature: feature_years_since_last_donation_past_5yr
Historical periods available: 5
No historical donation value: 6


In [44]:
# Document the meaning of each supported recency value
recency_value_definitions = pd.DataFrame(
    [
        {
            "recency_value": period,
            "meaning": (
                "Donated during the most recent completed fiscal year"
                if period == 0
                else (
                    f"Most recent donation was "
                    f"{period + 1} fiscal years ago"
                )
            ),
        }
        for period in range(historical_period_count)
    ]
    + [
        {
            "recency_value": NO_HISTORICAL_DONATION_RECENCY_VALUE,
            "meaning": (
                "No donation was found in the available "
                "historical window"
            ),
        }
    ]
)

display(recency_value_definitions.style.hide(axis="index"))

# Validate the recency values against the documentation
documented_recency_values = set(
    recency_value_definitions["recency_value"]
)

observed_recency_values = set(
    df[RECENCY_FEATURE_COLUMN].unique()
)

unexpected_recency_values = sorted(
    observed_recency_values - documented_recency_values
)

assert not unexpected_recency_values, (
    "Unexpected recency values found: "
    f"{unexpected_recency_values}"
)

print("Recency resolution validation passed.")

recency_value,meaning
0,Donated during the most recent completed fiscal year
1,Most recent donation was 2 fiscal years ago
2,Most recent donation was 3 fiscal years ago
3,Most recent donation was 4 fiscal years ago
4,Most recent donation was 5 fiscal years ago
6,No donation was found in the available historical window


Recency resolution validation passed.


## Donation Recency Resolution

The available dataset supports donation recency only at the fiscal year level.

The five historical donation columns contain annual fiscal year totals, while no individual donation dates, timestamps, or monthly donation fields are available. Therefore, recency cannot be calculated reliably in days or months.

`feature_years_since_last_donation_past_5yr` uses the following values:

- `0`: Donated during the most recent completed fiscal year
- `1`: Most recent donation was two fiscal years ago
- `2`: Most recent donation was three fiscal years ago
- `3`: Most recent donation was four fiscal years ago
- `4`: Most recent donation was five fiscal years ago
- `6`: No donation was found in the available historical window

The value `6` represents no observed donation within the five year historical window. It should not be interpreted as exactly six years since the individual last donated.

A day or month based recency feature was not created because the dataset does not support that level of precision.


In [45]:
# Create trend features
HISTORICAL_DONATION_COLUMNS_CHRONOLOGICAL = [
    "donation_5_fiscal_years_ago",
    "donation_4_fiscal_years_ago",
    "donation_3_fiscal_years_ago",
    "donation_2_fiscal_years_ago",
    "last_fiscal_year_donation",
]

RECENT_HISTORICAL_DONATION_COLUMNS = [
    "last_fiscal_year_donation",
    "donation_2_fiscal_years_ago",
]

OLDER_HISTORICAL_DONATION_COLUMNS = [
    "donation_3_fiscal_years_ago",
    "donation_4_fiscal_years_ago",
    "donation_5_fiscal_years_ago",
]

historical_donation_values = df[
    HISTORICAL_DONATION_COLUMNS_CHRONOLOGICAL
].to_numpy(dtype="float64")

recent_historical_average = df[
    RECENT_HISTORICAL_DONATION_COLUMNS
].mean(axis=1)

older_historical_average = df[
    OLDER_HISTORICAL_DONATION_COLUMNS
].mean(axis=1)


df["feature_recent_vs_older_donation_difference"] = (
    recent_historical_average
    - older_historical_average
)



df["feature_recent_vs_older_donation_ratio"] = np.divide(
    recent_historical_average,
    older_historical_average,
    out=np.full(
        len(df),
        np.nan,
        dtype="float64",
    ),
    where=older_historical_average.ne(0),
)


centered_time_positions = np.arange(
    len(HISTORICAL_DONATION_COLUMNS_CHRONOLOGICAL),
    dtype="float64",
)

centered_time_positions -= centered_time_positions.mean()

slope_denominator = np.square(
    centered_time_positions
).sum()

df["feature_past_5yr_donation_trend_slope"] = (
    historical_donation_values
    @ centered_time_positions
) / slope_denominator


df["feature_last_year_donation_amount_change"] = (
    df["last_fiscal_year_donation"]
    - df["donation_2_fiscal_years_ago"]
)


DONATION_VOLATILITY_FEATURE = (
    "feature_past_5yr_donation_std"
)

assert DONATION_VOLATILITY_FEATURE in df.columns, (
    f"{DONATION_VOLATILITY_FEATURE} was not found. "
    "Complete the historical donation summary feature task first."
)


historical_donation_average = df[
    HISTORICAL_DONATION_COLUMNS_CHRONOLOGICAL
].mean(axis=1)

df[
    "feature_past_5yr_donation_coefficient_of_variation"
] = np.divide(
    df[DONATION_VOLATILITY_FEATURE],
    historical_donation_average,
    out=np.zeros(
        len(df),
        dtype="float64",
    ),
    where=historical_donation_average.ne(0),
)

In [46]:
# Validate trend features
DONATION_TREND_FEATURES = [
    "feature_recent_vs_older_donation_difference",
    "feature_recent_vs_older_donation_ratio",
    "feature_past_5yr_donation_trend_slope",
    "feature_last_year_donation_amount_change",
    "feature_past_5yr_donation_std",
    "feature_past_5yr_donation_coefficient_of_variation",
]

trend_feature_validation = pd.DataFrame(
    {
        "missing_values": (
            df[DONATION_TREND_FEATURES]
            .isna()
            .sum()
        ),
        "infinite_values": [
            np.isinf(df[column]).sum()
            for column in DONATION_TREND_FEATURES
        ],
    }
)

display(trend_feature_validation)

,missing_values,infinite_values
feature_recent_vs_older_donation_difference,0,0
feature_recent_vs_older_donation_ratio,28497,0
feature_past_5yr_donation_trend_slope,0,0
feature_last_year_donation_amount_change,0,0
feature_past_5yr_donation_std,0,0
feature_past_5yr_donation_coefficient_of_variation,0,0


In [47]:
ratio_denominator_zero = (
    older_historical_average == 0
)

expected_missing_ratios = (
    ratio_denominator_zero.sum()
)

actual_missing_ratios = df[
    "feature_recent_vs_older_donation_ratio"
].isna().sum()

assert (
    actual_missing_ratios
    == expected_missing_ratios
), (
    "Unexpected number of missing recent-to-older "
    "donation ratios."
)

non_ratio_trend_features = [
    feature
    for feature in DONATION_TREND_FEATURES
    if feature
    != "feature_recent_vs_older_donation_ratio"
]

assert (
    df[non_ratio_trend_features]
    .isna()
    .sum()
    .sum()
    == 0
), "Unexpected missing values found."

assert not np.isinf(
    df[DONATION_TREND_FEATURES]
).any().any(), "Infinite trend feature values found."

assert (
    df[
        "feature_past_5yr_donation_coefficient_of_variation"
    ]
    >= 0
).all(), "Negative coefficients of variation found."

print("Donation trend feature validation passed.")
print(
    "Undefined recent-to-older ratios: "
    f"{actual_missing_ratios:,}"
)

Donation trend feature validation passed.
Undefined recent-to-older ratios: 28,497


In [48]:
# Trend feature distribution
trend_feature_summary = (
    df[DONATION_TREND_FEATURES]
    .describe()
    .T
)

trend_feature_summary["nonzero_records"] = [
    df[column].fillna(0).ne(0).sum()
    for column in DONATION_TREND_FEATURES
]

display(
    trend_feature_summary.style.format(
        lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x
    )
)

,count,mean,std,min,25%,50%,75%,max,nonzero_records
feature_recent_vs_older_donation_difference,"34,403",154.87,"27,712.1","-500,000",0,0,0,"4,933,408.67","9,073"
feature_recent_vs_older_donation_ratio,"5,906",3.99,53.06,0,0,0,0,"2,550","1,417"
feature_past_5yr_donation_trend_slope,"34,403",54.27,"11,144.84","-275,000",0,0,0,"1,999,973.6","7,779"
feature_last_year_donation_amount_change,"34,403",282.27,"54,608.39","-1,432,960",0,0,0,"9,999,800","4,582"
feature_past_5yr_donation_std,"34,403",236.14,"22,261.85",0,0,0,0.4,"3,980,724.03","9,087"
feature_past_5yr_donation_coefficient_of_variation,"34,403",0.5,0.84,0,0,0,1.24,2,"9,087"


In [49]:
# Review direction of 2 primary trend measures
trend_direction_summary = pd.DataFrame(
    {
        "recent_vs_older_difference": {
            "increasing": (
                df[
                    "feature_recent_vs_older_donation_difference"
                ] > 0
            ).sum(),
            "stable": (
                df[
                    "feature_recent_vs_older_donation_difference"
                ] == 0
            ).sum(),
            "decreasing": (
                df[
                    "feature_recent_vs_older_donation_difference"
                ] < 0
            ).sum(),
        },
        "five_year_slope": {
            "increasing": (
                df[
                    "feature_past_5yr_donation_trend_slope"
                ] > 0
            ).sum(),
            "stable": (
                df[
                    "feature_past_5yr_donation_trend_slope"
                ] == 0
            ).sum(),
            "decreasing": (
                df[
                    "feature_past_5yr_donation_trend_slope"
                ] < 0
            ).sum(),
        },
    }
)

display(trend_direction_summary)

,recent_vs_older_difference,five_year_slope
increasing,4071,4248
stable,25330,26624
decreasing,5002,3531


## Historical Donation Trends

Donation trend features were created using the five completed fiscal years before the target outcome period. The historical donation columns were arranged from the oldest period to the most recent period so that positive values consistently represent increasing giving over time.

The following features were created:
- `feature_recent_vs_older_donation_difference` compares the average donation amount during the two most recent historical years with the average during the three older historical years. Positive values indicate that recent giving was higher, while negative values indicate that older giving was higher.
- `feature_recent_vs_older_donation_ratio` measures recent average giving relative to older average giving. The ratio is undefined when the older historical average is zero, resulting in 28,497 missing values. These values were retained as missing rather than assigning an arbitrary ratio that could misrepresent the donor's history.
- `feature_past_5yr_donation_trend_slope` estimates the linear direction of donation amounts across the five historical fiscal years. Positive values indicate an upward trend, negative values indicate a downward trend, and zero indicates no linear change.
- `feature_last_year_donation_amount_change` measures the difference between the most recent completed fiscal year and the preceding fiscal year.
- `feature_past_5yr_donation_std`, created during the historical donation summary task, is reused as the absolute donation volatility measure.
- `feature_past_5yr_donation_coefficient_of_variation` measures donation volatility relative to average historical giving. Individuals with no historical donations receive a value of zero because both their average donation and donation volatility are zero.

Based on the recent versus older donation difference, 4,071 individuals showed increasing donation amounts, 5,002 showed decreasing amounts, and 25,330 showed no change. Based on the five year linear slope, 4,248 individuals showed an increasing trend, 3,531 showed a decreasing trend, and 26,624 had a stable trend.

The high number of stable values reflects the large proportion of individuals with no donations or unchanged annual donation amounts during the historical window. The trend features are also highly skewed because a small number of donors contributed extremely large amounts.

The recent versus older difference and five year trend slope are available for every record, while the ratio is available only when the older historical average is greater than zero. 

No current fiscal year donation information was used to create these features.


In [50]:
# Create year over year trajectory features
LAST_VS_PREVIOUS_DIFFERENCE_FEATURE = (
    "feature_last_year_donation_amount_change"
)

RECENT_VS_OLDER_DIFFERENCE_FEATURE = (
    "feature_recent_vs_older_donation_difference"
)

RECENT_VS_OLDER_RATIO_FEATURE = (
    "feature_recent_vs_older_donation_ratio"
)

required_existing_features = [
    LAST_VS_PREVIOUS_DIFFERENCE_FEATURE,
    RECENT_VS_OLDER_DIFFERENCE_FEATURE,
    RECENT_VS_OLDER_RATIO_FEATURE,
]

missing_existing_features = [
    feature
    for feature in required_existing_features
    if feature not in df.columns
]

assert not missing_existing_features, (
    "Required trajectory features are missing: "
    f"{missing_existing_features}"
)


df[
    "feature_previous_year_donation_zero_flag"
] = (
    df["donation_2_fiscal_years_ago"] == 0
).astype("int8")


df[
    "feature_last_vs_previous_donation_ratio"
] = (
    df["last_fiscal_year_donation"]
    / (
        df["donation_2_fiscal_years_ago"]
        + 1
    )
)


df[
    "feature_log1p_last_vs_previous_donation_ratio"
] = np.log1p(
    df[
        "feature_last_vs_previous_donation_ratio"
    ]
)

In [51]:
# Validate year over year features
YEAR_OVER_YEAR_TRAJECTORY_FEATURES = [
    "feature_last_year_donation_amount_change",
    "feature_last_vs_previous_donation_ratio",
    "feature_log1p_last_vs_previous_donation_ratio",
    "feature_previous_year_donation_zero_flag",
    "feature_recent_vs_older_donation_difference",
    "feature_recent_vs_older_donation_ratio",
]

trajectory_feature_validation = pd.DataFrame(
    {
        "missing_values": (
            df[YEAR_OVER_YEAR_TRAJECTORY_FEATURES]
            .isna()
            .sum()
        ),
        "infinite_values": [
            np.isinf(df[column]).sum()
            for column in YEAR_OVER_YEAR_TRAJECTORY_FEATURES
        ],
    }
)

display(
    trajectory_feature_validation.style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
    .format(
        lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x
    )
)

,missing_values,infinite_values
feature_last_year_donation_amount_change,0,0
feature_last_vs_previous_donation_ratio,0,0
feature_log1p_last_vs_previous_donation_ratio,0,0
feature_previous_year_donation_zero_flag,0,0
feature_recent_vs_older_donation_difference,0,0
feature_recent_vs_older_donation_ratio,"28,497",0


In [52]:
non_nullable_trajectory_features = [
    "feature_last_year_donation_amount_change",
    "feature_last_vs_previous_donation_ratio",
    "feature_log1p_last_vs_previous_donation_ratio",
    "feature_previous_year_donation_zero_flag",
    "feature_recent_vs_older_donation_difference",
]

assert (
    df[non_nullable_trajectory_features]
    .isna()
    .sum()
    .sum()
    == 0
), "Unexpected missing trajectory feature values found."

assert not np.isinf(
    df[YEAR_OVER_YEAR_TRAJECTORY_FEATURES]
).any().any(), "Infinite trajectory feature values found."

assert (
    df["feature_last_vs_previous_donation_ratio"]
    >= 0
).all(), "Negative donation ratios found."

assert (
    df["feature_log1p_last_vs_previous_donation_ratio"]
    >= 0
).all(), "Negative log-transformed ratios found."

assert set(
    df["feature_previous_year_donation_zero_flag"].unique()
).issubset({0, 1}), (
    "Invalid denominator-zero indicator values found."
)

print("Year-over-year trajectory validation passed.")

Year-over-year trajectory validation passed.


In [53]:
# Review ratio distribution
trajectory_ratio_summary = (
    df[
        [
            "feature_last_vs_previous_donation_ratio",
            "feature_log1p_last_vs_previous_donation_ratio",
        ]
    ]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999,
        ]
    )
    .T
)

trajectory_ratio_summary["nonzero_records"] = [
    df[column].ne(0).sum()
    for column in trajectory_ratio_summary.index
]

display(
    trajectory_ratio_summary.style
        .set_properties(**{"text-align": "center"})
        .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
        .format(
        lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x
    )
)

,count,mean,std,min,50%,75%,90%,95%,99%,99.9%,max,nonzero_records
feature_last_vs_previous_donation_ratio,"34,403",31.66,"2,772.45",0,0,0,0,2.62,200,"1,981.51","500,000","2,358"
feature_log1p_last_vs_previous_donation_ratio,"34,403",0.24,1.02,0,0,0,0,1.29,5.3,7.59,13.12,"2,358"


In [54]:
previous_year_zero_summary = pd.DataFrame(
    {
        "record_count": (
            df[
                "feature_previous_year_donation_zero_flag"
            ]
            .value_counts()
            .sort_index()
        ),
        "percentage": (
            df[
                "feature_previous_year_donation_zero_flag"
            ]
            .value_counts(
                normalize=True
            )
            .sort_index()
            .mul(100)
        ),
    }
)

display(previous_year_zero_summary.style
        .set_properties(**{"text-align": "center"})
        .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
        .format({
        "percentage": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') + "%" if isinstance(x, (int, float)) else x,
        "record_count": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x
        })
)

,record_count,percentage
feature_previous_year_donation_zero_flag,,
0,"2,463",7.16%
1,"31,940",92.84%


## Year-over-Year Donation Value Trajectories

Year-over-year trajectory features were used to measure changes in donation value between historical periods.

Several trajectory measures were already created during the historical donation trend task and were reused rather than duplicated:
- `feature_last_year_donation_amount_change` measures the difference between the most recent historical donation amount and the preceding historical donation amount.
- `feature_recent_vs_older_donation_difference` compares the average donation amount during the two most recent historical years with the average during the three older historical years.
- `feature_recent_vs_older_donation_ratio` measures recent average giving relative to older average giving.

The following additional features were created:
- `feature_last_vs_previous_donation_ratio` compares the most recent historical donation amount with the preceding year's donation amount. A constant of one was added to the denominator to prevent division by zero.
- `feature_previous_year_donation_zero_flag` identifies records where the preceding year's donation amount was zero.
- `feature_log1p_last_vs_previous_donation_ratio` applies a logarithmic transformation to reduce the influence of extremely large ratio values.

The raw last versus previous donation ratio was highly skewed. Its median, 75th percentile, and 90th percentile were all zero, while its maximum value reached 500,000. The logarithmic transformation reduced the maximum to 13.12 and compressed the upper tail while preserving the relative ordering of the observations.

The preceding historical donation amount was zero for 31,940 records, representing 92.84% of the dataset. This confirms that the denominator-zero indicator is important because the adjusted ratio frequently relies on the added constant.

The raw ratio, log-transformed ratio, and absolute difference were retained because they capture different aspects of donation behavior. The raw ratio preserves the original proportional change, the logarithmic version reduces extreme skew, and the difference retains the monetary magnitude of the change.

No current fiscal year donation information was used to create these features.


In [55]:
# Donation features selected for log transformation
DONATION_TRANSFORMATION_MAPPING = {
    "feature_past_5yr_total_donation":
        "feature_log_past_5yr_total_donation",
    "feature_past_5yr_average_donation":
        "feature_log_past_5yr_average_donation",
    "feature_past_5yr_max_donation":
        "feature_log_past_5yr_max_donation",
    "feature_most_recent_positive_donation":
        "feature_log_most_recent_positive_donation",
}

missing_source_features = [
    feature
    for feature in DONATION_TRANSFORMATION_MAPPING
    if feature not in df.columns
]

assert not missing_source_features, (
    "Required source features are missing: "
    f"{missing_source_features}"
)

for source_feature, transformed_feature in (
    DONATION_TRANSFORMATION_MAPPING.items()
):
    df[transformed_feature] = np.log1p(
        df[source_feature]
    )

In [56]:
# Validate transformations
TRANSFORMED_DONATION_FEATURES = list(
    DONATION_TRANSFORMATION_MAPPING.values()
)

transformation_validation = pd.DataFrame(
    {
        "missing_values": (
            df[TRANSFORMED_DONATION_FEATURES]
            .isna()
            .sum()
        ),
        "infinite_values": [
            np.isinf(df[column]).sum()
            for column in TRANSFORMED_DONATION_FEATURES
        ],
        "negative_values": [
            (df[column] < 0).sum()
            for column in TRANSFORMED_DONATION_FEATURES
        ],
    }
)

display(transformation_validation.style.set_properties(**{"text-align": "center"}))

assert (
    transformation_validation.values.sum() == 0
), "Transformation validation failed."

print("Donation transformation validation passed.")

,missing_values,infinite_values,negative_values
feature_log_past_5yr_total_donation,0,0,0
feature_log_past_5yr_average_donation,0,0,0
feature_log_past_5yr_max_donation,0,0,0
feature_log_most_recent_positive_donation,0,0,0


Donation transformation validation passed.


In [57]:
# Compare og vs log features
transformation_summary = pd.DataFrame(
    {
        "original_max": [
            df[source].max()
            for source in DONATION_TRANSFORMATION_MAPPING
        ],
        "log_max": [
            df[target].max()
            for target in TRANSFORMED_DONATION_FEATURES
        ],
        "original_mean": [
            df[source].mean()
            for source in DONATION_TRANSFORMATION_MAPPING
        ],
        "log_mean": [
            df[target].mean()
            for target in TRANSFORMED_DONATION_FEATURES
        ],
    },
    index=DONATION_TRANSFORMATION_MAPPING.keys(),
)

display(transformation_summary.style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
    .format({
        "original_max": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x,
        "log_max": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x,
        "original_mean": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x,
        "log_mean": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x
    })
)

,original_max,log_max,original_mean,log_mean
feature_past_5yr_total_donation,"10,200,274",16.14,723.44,1.06
feature_past_5yr_average_donation,"2,040,054.8",14.53,144.69,0.69
feature_past_5yr_max_donation,"10,000,000",16.12,596.12,1.04
feature_most_recent_positive_donation,"10,000,000",16.12,440.69,1


In [58]:
comparison_summary = []

for source, target in DONATION_TRANSFORMATION_MAPPING.items():
    original = df[source]
    transformed = df[target]

    comparison_summary.append(
        {
            "feature": source,
            "original_max": original.max(),
            "log_max": transformed.max(),
            "original_std": original.std(),
            "log_std": transformed.std(),
        }
    )

display(pd.DataFrame(comparison_summary).style.hide(axis="index")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
    .format({
        "original_max": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x,
        "log_max": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x,
        "original_std": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x,
        "log_std": lambda x: f"{x:,.2f}".rstrip('0').rstrip('.') if isinstance(x, (int, float)) else x
    })
       
)

feature,original_max,log_max,original_std,log_std
feature_past_5yr_total_donation,"10,200,274",16.14,"58,342.93",2.03
feature_past_5yr_average_donation,"2,040,054.8",14.53,"11,668.59",1.44
feature_past_5yr_max_donation,"10,000,000",16.12,"55,992.71",1.98
feature_most_recent_positive_donation,"10,000,000",16.12,"54,180.69",1.92


## Skew Resistant Donation Transformations

Several donation amount features were transformed using `np.log1p` to reduce the influence of extreme values while preserving zero donation amounts.

The following transformed features were created:
- `feature_log_past_5yr_total_donation`
- `feature_log_past_5yr_average_donation`
- `feature_log_past_5yr_max_donation`
- `feature_log_most_recent_positive_donation`

The original donation features were retained alongside the transformed versions. This allows later models to determine whether the original monetary values or the skew resistant representations provide stronger predictive performance.

- The log transformation substantially reduced the scale and variability of the donation features
  - The maximum five year total donation decreased from 10,200,274 to 16.14
  - The standard deviation decreased from 58,342.93 to 2.03
- Similar reductions were observed for:
  - `feature_log_past_5yr_average_donation`
  - `feature_log_past_5yr_max_donation`
  - `feature_log_most_recent_positive_donation`
- All transformed features passed validation with no missing, infinite, or negative values

No major donors were removed or manually capped because the extreme values may represent legitimate high-value donors.

In [59]:
# Validate source engagement columns
ENGAGEMENT_SOURCE_COLUMNS = [
    "is_alumnus_flag",
    "is_parent_flag",
    "has_involvement_flag",
    "has_email_flag",
]

missing_engagement_source_columns = [
    column
    for column in ENGAGEMENT_SOURCE_COLUMNS
    if column not in df.columns
]

assert not missing_engagement_source_columns, (
    "Required engagement source columns are missing: "
    f"{missing_engagement_source_columns}"
)

assert (
    df[ENGAGEMENT_SOURCE_COLUMNS]
    .isna()
    .sum()
    .sum()
    == 0
), "Missing values found in engagement source columns."

for column in ENGAGEMENT_SOURCE_COLUMNS:
    assert set(
        df[column].unique()
    ).issubset({0, 1}), (
        f"Unexpected values found in {column}."
    )

In [60]:
# Summarize source engagement indicators
engagement_source_summary = pd.DataFrame(
    {
        "record_count": (
            df[ENGAGEMENT_SOURCE_COLUMNS]
            .sum()
            .astype(int)
        ),
        "percentage": (
            df[ENGAGEMENT_SOURCE_COLUMNS]
            .mean()
            .mul(100)
        ),
    }
)

display(engagement_source_summary
    .style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "left")],
    }])
    .format({
        "record_count": "{:,.0f}",
        "percentage": "{:,.2f}%",
    })
)

,record_count,percentage
is_alumnus_flag,"8,418",24.47%
is_parent_flag,"2,694",7.83%
has_involvement_flag,"7,972",23.17%
has_email_flag,"11,258",32.72%


In [61]:
# Create engagement score
df["feature_engagement_score"] = (
    df[ENGAGEMENT_SOURCE_COLUMNS]
    .sum(axis=1)
    .astype("int8")
)


df["feature_has_any_engagement_flag"] = (
    df["feature_engagement_score"] > 0
).astype("int8")


df["feature_high_engagement_flag"] = (
    df["feature_engagement_score"] >= 2
).astype("int8")


df["feature_alumnus_involvement_interaction"] = (
    df["is_alumnus_flag"]
    * df["has_involvement_flag"]
).astype("int8")


df["feature_email_involvement_interaction"] = (
    df["has_email_flag"]
    * df["has_involvement_flag"]
).astype("int8")

In [62]:
# Validate engagement features
ENGAGEMENT_FEATURES = [
    "feature_engagement_score",
    "feature_has_any_engagement_flag",
    "feature_high_engagement_flag",
    "feature_alumnus_involvement_interaction",
    "feature_email_involvement_interaction",
]

BINARY_ENGAGEMENT_FEATURES = [
    "feature_has_any_engagement_flag",
    "feature_high_engagement_flag",
    "feature_alumnus_involvement_interaction",
    "feature_email_involvement_interaction",
]


engagement_feature_validation = pd.DataFrame(
    {
        "missing_values": (
            df[ENGAGEMENT_FEATURES]
            .isna()
            .sum()
        ),
        "minimum": [
            df[column].min()
            for column in ENGAGEMENT_FEATURES
        ],
        "maximum": [
            df[column].max()
            for column in ENGAGEMENT_FEATURES
        ],
    }
)

display(engagement_feature_validation
    .style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "left")],
    }])
)


assert (
    df[ENGAGEMENT_FEATURES]
    .isna()
    .sum()
    .sum()
    == 0
), "Missing engagement feature values found."

assert df["feature_engagement_score"].between(
    0,
    len(ENGAGEMENT_SOURCE_COLUMNS),
).all(), "Engagement score is outside the expected range."

for column in BINARY_ENGAGEMENT_FEATURES:
    assert set(
        df[column].unique()
    ).issubset({0, 1}), (
        f"Unexpected binary values found in {column}."
    )


# Validate engagement score
expected_engagement_score = (
    df[ENGAGEMENT_SOURCE_COLUMNS]
    .sum(axis=1)
    .astype("int8")
)

assert (
    df["feature_engagement_score"]
    == expected_engagement_score
).all(), "Engagement score validation failed."



assert (
    df["feature_has_any_engagement_flag"]
    == (
        df["feature_engagement_score"] > 0
    ).astype("int8")
).all(), "Any engagement flag validation failed."

assert (
    df["feature_high_engagement_flag"]
    == (
        df["feature_engagement_score"] >= 2
    ).astype("int8")
).all(), "High engagement flag validation failed."



assert (
    df["feature_alumnus_involvement_interaction"]
    == (
        df["is_alumnus_flag"]
        * df["has_involvement_flag"]
    )
).all(), (
    "Alumnus involvement interaction validation failed."
)

assert (
    df["feature_email_involvement_interaction"]
    == (
        df["has_email_flag"]
        * df["has_involvement_flag"]
    )
).all(), (
    "Email involvement interaction validation failed."
)


assert (
    df["feature_alumnus_involvement_interaction"].sum()
    <= df["is_alumnus_flag"].sum()
), (
    "Alumnus involvement interaction count exceeds "
    "the alumnus count."
)

assert (
    df["feature_alumnus_involvement_interaction"].sum()
    <= df["has_involvement_flag"].sum()
), (
    "Alumnus involvement interaction count exceeds "
    "the involvement count."
)

assert (
    df["feature_email_involvement_interaction"].sum()
    <= df["has_email_flag"].sum()
), (
    "Email involvement interaction count exceeds "
    "the email count."
)

assert (
    df["feature_email_involvement_interaction"].sum()
    <= df["has_involvement_flag"].sum()
), (
    "Email involvement interaction count exceeds "
    "the involvement count."
)

print("Engagement feature validation passed.")

,missing_values,minimum,maximum
feature_engagement_score,0,0,4
feature_has_any_engagement_flag,0,0,1
feature_high_engagement_flag,0,0,1
feature_alumnus_involvement_interaction,0,0,1
feature_email_involvement_interaction,0,0,1


Engagement feature validation passed.


In [63]:
# Distribution of engagement scores
engagement_score_summary = pd.DataFrame(
    {
        "record_count": (
            df["feature_engagement_score"]
            .value_counts()
            .sort_index()
        ),
        "percentage": (
            df["feature_engagement_score"]
            .value_counts(normalize=True)
            .sort_index()
            .mul(100)
        ),
    }
)

display(engagement_score_summary
    .style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "center")],
    }])
    .format({
        "record_count": "{:,.0f}",
        "percentage": "{:,.2f}%",
    })
)

,record_count,percentage
feature_engagement_score,,
0,"16,862",49.01%
1,"8,903",25.88%
2,"4,580",13.31%
3,"3,953",11.49%
4,105,0.31%


In [64]:
# Summarize binary engagement features
engagement_flag_summary = pd.DataFrame(
    {
        "record_count": [
            int(df[column].sum())
            for column in BINARY_ENGAGEMENT_FEATURES
        ],
        "percentage": [
            df[column].mean() * 100
            for column in BINARY_ENGAGEMENT_FEATURES
        ],
    },
    index=BINARY_ENGAGEMENT_FEATURES,
)

display(
    engagement_flag_summary.style
        .set_properties(**{"text-align": "center"})
        .set_table_styles(
            [
                {
                    "selector": "th",
                    "props": [("text-align", "left")],
                }
            ]
        )
        .format(
            {
                "record_count": "{:,.0f}",
                "percentage": "{:,.2f}%",
            }
        )
)

,record_count,percentage
feature_has_any_engagement_flag,"17,541",50.99%
feature_high_engagement_flag,"8,638",25.11%
feature_alumnus_involvement_interaction,"4,674",13.59%
feature_email_involvement_interaction,"4,706",13.68%


## Donor Engagement Features

Combined engagement features were created from alumni, parent, involvement, and email indicators. The original binary indicators were retained alongside the engineered features so later models can evaluate both the individual and combined representations.

The source engagement indicators showed:

- 8,418 individuals, or 24.47%, were alumni
- 2,694 individuals, or 7.83%, were parents
- 7,972 individuals, or 23.17%, had recorded involvement
- 11,258 individuals, or 32.72%, had an email indicator

The following features were created:
- `feature_engagement_score` counts the number of active indicators across alumni status, parent status, involvement status, and email availability. Values range from zero through four.
- `feature_has_any_engagement_flag` identifies individuals with at least one active engagement indicator.
- `feature_high_engagement_flag` identifies individuals with two or more active engagement indicators.
- `feature_alumnus_involvement_interaction` identifies individuals who were both alumni and recorded as involved.
- `feature_email_involvement_interaction` identifies individuals who had both an email indicator and recorded involvement.

The engagement score distribution showed:

- 16,862 individuals, or 49.01%, had no active engagement indicators
- 8,903 individuals, or 25.88%, had one active indicator
- 4,580 individuals, or 13.31%, had two active indicators
- 3,953 individuals, or 11.49%, had three active indicators
- 105 individuals, or 0.31%, had all four indicators

Overall, 17,541 individuals, or 50.99%, had at least one engagement indicator. A total of 8,638 individuals, or 25.11%, met the high engagement threshold of two or more active indicators.

The interaction features identified 4,674 individuals, or 13.59%, who were both alumni and involved. They also identified 4,706 individuals, or 13.68%, who had both an email indicator and recorded involvement.

No contactability feature was created because the dataset does not contain complete postal addresses. `preferred_address_type` describes only the preferred address category and does not confirm that a valid or deliverable address is available.

In [65]:
# Define core & expanded engagement source columns
CORE_ENGAGEMENT_SOURCE_COLUMNS = [
    "is_alumnus_flag",
    "has_involvement_flag",
    "has_email_flag",
]

EXPANDED_ENGAGEMENT_SOURCE_COLUMNS = [
    "is_alumnus_flag",
    "is_parent_flag",
    "has_involvement_flag",
    "has_email_flag",
]


required_engagement_index_columns = (
    EXPANDED_ENGAGEMENT_SOURCE_COLUMNS
    + [
        "feature_engagement_score",
        "feature_high_engagement_flag",
    ]
)

missing_engagement_index_columns = [
    column
    for column in required_engagement_index_columns
    if column not in df.columns
]

assert not missing_engagement_index_columns, (
    "Required engagement index columns are missing: "
    f"{missing_engagement_index_columns}"
)

In [66]:
# Create core engagement index
df["feature_core_engagement_score"] = (
    df[CORE_ENGAGEMENT_SOURCE_COLUMNS]
    .sum(axis=1)
    .astype("int8")
)

df["feature_core_high_engagement_flag"] = (
    df["feature_core_engagement_score"] >= 2
).astype("int8")

In [67]:
# Validate engagement indexes
ENGAGEMENT_INDEX_FEATURES = [
    "feature_core_engagement_score",
    "feature_engagement_score",
    "feature_core_high_engagement_flag",
    "feature_high_engagement_flag",
]

engagement_index_validation = pd.DataFrame(
    {
        "missing_values": (
            df[ENGAGEMENT_INDEX_FEATURES]
            .isna()
            .sum()
        ),
        "minimum": [
            df[column].min()
            for column in ENGAGEMENT_INDEX_FEATURES
        ],
        "maximum": [
            df[column].max()
            for column in ENGAGEMENT_INDEX_FEATURES
        ],
    }
)

display(engagement_index_validation.style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "left")],
    }])
)


assert (
    df[ENGAGEMENT_INDEX_FEATURES]
    .isna()
    .sum()
    .sum()
    == 0
), "Missing engagement index values found."


assert df["feature_core_engagement_score"].between(
    0,
    len(CORE_ENGAGEMENT_SOURCE_COLUMNS),
).all(), "Core engagement score is outside the expected range."

assert df["feature_engagement_score"].between(
    0,
    len(EXPANDED_ENGAGEMENT_SOURCE_COLUMNS),
).all(), "Expanded engagement score is outside the expected range."


expected_core_engagement_score = (
    df[CORE_ENGAGEMENT_SOURCE_COLUMNS]
    .sum(axis=1)
    .astype("int8")
)

expected_expanded_engagement_score = (
    df[EXPANDED_ENGAGEMENT_SOURCE_COLUMNS]
    .sum(axis=1)
    .astype("int8")
)

assert (
    df["feature_core_engagement_score"]
    == expected_core_engagement_score
).all(), "Core engagement score validation failed."

assert (
    df["feature_engagement_score"]
    == expected_expanded_engagement_score
).all(), "Expanded engagement score validation failed."


assert (
    df["feature_engagement_score"]
    == (
        df["feature_core_engagement_score"]
        + df["is_parent_flag"]
    )
).all(), (
    "Expanded engagement score does not equal the "
    "core score plus parental status."
)


assert set(
    df["feature_core_high_engagement_flag"].unique()
).issubset({0, 1}), (
    "Unexpected values found in the core high engagement flag."
)

assert set(
    df["feature_high_engagement_flag"].unique()
).issubset({0, 1}), (
    "Unexpected values found in the expanded high engagement flag."
)


assert (
    df["feature_core_high_engagement_flag"]
    == (
        df["feature_core_engagement_score"] >= 2
    ).astype("int8")
).all(), "Core high engagement flag validation failed."

assert (
    df["feature_high_engagement_flag"]
    == (
        df["feature_engagement_score"] >= 2
    ).astype("int8")
).all(), "Expanded high engagement flag validation failed."


assert (
    df["feature_engagement_score"]
    >= df["feature_core_engagement_score"]
).all(), (
    "Expanded engagement score is below the core score "
    "for at least one record."
)

assert (
    df["feature_high_engagement_flag"]
    >= df["feature_core_high_engagement_flag"]
).all(), (
    "Expanded high engagement classification is below "
    "the core classification for at least one record."
)

print("Engagement index validation passed.")

,missing_values,minimum,maximum
feature_core_engagement_score,0,0,3
feature_engagement_score,0,0,4
feature_core_high_engagement_flag,0,0,1
feature_high_engagement_flag,0,0,1


Engagement index validation passed.


In [68]:
# Compare engagement score distributions
core_engagement_distribution = (
    df["feature_core_engagement_score"]
    .value_counts()
    .sort_index()
    .reindex(range(4), fill_value=0)
)

expanded_engagement_distribution = (
    df["feature_engagement_score"]
    .value_counts()
    .sort_index()
    .reindex(range(5), fill_value=0)
)


engagement_index_distribution_comparison = pd.DataFrame(
    {
        "core_record_count": (
            core_engagement_distribution
            .reindex(range(5), fill_value=0)
        ),
        "core_percentage": (
            core_engagement_distribution
            .reindex(range(5), fill_value=0)
            .div(len(df))
            .mul(100)
        ),
        "expanded_record_count": (
            expanded_engagement_distribution
        ),
        "expanded_percentage": (
            expanded_engagement_distribution
            .div(len(df))
            .mul(100)
        ),
    }
)

engagement_index_distribution_comparison.index.name = (
    "engagement_score"
)

display(engagement_index_distribution_comparison.style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "center")],
    }])
    .format({
        "core_record_count": "{:,.0f}",
        "core_percentage": "{:,.2f}%",
        "expanded_record_count": "{:,.0f}",
        "expanded_percentage": "{:,.2f}%",
    })
)

,core_record_count,core_percentage,expanded_record_count,expanded_percentage
engagement_score,,,,
0,"17,652",51.31%,"16,862",49.01%
1,"9,622",27.97%,"8,903",25.88%
2,"3,361",9.77%,"4,580",13.31%
3,"3,768",10.95%,"3,953",11.49%
4,0,0.00%,105,0.31%


In [69]:
# Compare high engagement classifications
high_engagement_index_summary = pd.DataFrame(
    {
        "record_count": [
            int(
                df[
                    "feature_core_high_engagement_flag"
                ].sum()
            ),
            int(
                df[
                    "feature_high_engagement_flag"
                ].sum()
            ),
        ],
        "percentage": [
            (
                df[
                    "feature_core_high_engagement_flag"
                ].mean()
                * 100
            ),
            (
                df[
                    "feature_high_engagement_flag"
                ].mean()
                * 100
            ),
        ],
    },
    index=[
        "Core engagement index",
        "Expanded engagement index",
    ],
)

display(high_engagement_index_summary.style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "left")],
    }])
    .format({
        "record_count": "{:,.0f}",
        "percentage": "{:,.2f}%",
    })
)

,record_count,percentage
Core engagement index,"7,129",20.72%
Expanded engagement index,"8,638",25.11%


In [70]:
# Changes to engagement classification by parental status
parent_status_changes_high_engagement = (
    (
        df["feature_core_high_engagement_flag"] == 0
    )
    & (
        df["feature_high_engagement_flag"] == 1
    )
)

parental_status_effect_summary = pd.DataFrame(
    {
        "record_count": [
            int(df["is_parent_flag"].sum()),
            int(parent_status_changes_high_engagement.sum()),
        ],
        "percentage": [
            df["is_parent_flag"].mean() * 100,
            (
                parent_status_changes_high_engagement
                .mean()
                * 100
            ),
        ],
    },
    index=[
        "Records with parental status",
        (
            "Records moved into high engagement "
            "by parental status"
        ),
    ],
)

display(parental_status_effect_summary.style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "left")],
    }])
    .format({
        "record_count": "{:,.0f}",
        "percentage": "{:,.2f}%",
    })
)

,record_count,percentage
Records with parental status,"2,694",7.83%
Records moved into high engagement by parental status,"1,509",4.39%


## Comparing Core and Expanded Engagement Indexes

Two engagement Indexes were compared to evaluate whether parental status provides additional information beyond the primary engagement indicators.

The core engagement index includes alumni status, involvement status, and email availability. Scores range from `0` through `3`.

The expanded engagement index includes the same indicators while also incorporating parental status. Scores range from `0` through `4`. This expanded index is the same as the engagement score created in the previous task.

The core engagement score distribution showed:

- 17,652 individuals, or 51.31%, had a score of zero
- 9,622 individuals, or 27.97%, had a score of one
- 3,361 individuals, or 9.77%, had a score of two
- 3,768 individuals, or 10.95%, had the maximum score of three

The expanded engagement score distribution showed:

- 16,862 individuals, or 49.01%, had a score of zero
- 8,903 individuals, or 25.88%, had a score of one
- 4,580 individuals, or 13.31%, had a score of two
- 3,953 individuals, or 11.49%, had a score of three
- 105 individuals, or 0.31%, had the maximum score of four

Using a threshold of two or more engagement indicators, the core engagement index classified 7,129 individuals, or 20.72%, as highly engaged. Including parental status increased this total to 8,638 individuals, or 25.11%.

A total of 2,694 individuals, or 7.83%, were identified as parents. Among these, 1,509 individuals, or 4.39% of the dataset, moved from below the high-engagement threshold to the high-engagement category after parental status was included.

In [71]:
# Validate and summarize donor age
AGE_SOURCE_COLUMN = "donor_age"

assert AGE_SOURCE_COLUMN in df.columns, (
    f"Required age source column is missing: "
    f"{AGE_SOURCE_COLUMN}"
)

assert pd.api.types.is_numeric_dtype(
    df[AGE_SOURCE_COLUMN]
), "Donor age must be numeric."

assert (
    df.loc[
        df[AGE_SOURCE_COLUMN].notna(),
        AGE_SOURCE_COLUMN,
    ]
    >= 16
).all(), "Donor ages below 16 were found."


age_distribution_summary = pd.DataFrame(
    {
        "record_count": [
            df[AGE_SOURCE_COLUMN].count()
        ],
        "missing_values": [
            df[AGE_SOURCE_COLUMN].isna().sum()
        ],
        "minimum": [
            df[AGE_SOURCE_COLUMN].min()
        ],
        "first_quartile": [
            df[AGE_SOURCE_COLUMN].quantile(0.25)
        ],
        "median": [
            df[AGE_SOURCE_COLUMN].median()
        ],
        "mean": [
            df[AGE_SOURCE_COLUMN].mean()
        ],
        "third_quartile": [
            df[AGE_SOURCE_COLUMN].quantile(0.75)
        ],
        "maximum": [
            df[AGE_SOURCE_COLUMN].max()
        ],
    },
    index=[AGE_SOURCE_COLUMN],
)

display(age_distribution_summary.style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "center")],
    }])
    .format({
        "record_count": "{:,.0f}",
        "missing_values": "{:,.0f}",
        "minimum": "{:,.0f}",
        "first_quartile": "{:,.2f}",
        "median": "{:,.2f}",
        "mean": "{:,.2f}",
        "third_quartile": "{:,.2f}",
        "maximum": "{:,.0f}",
    })
)

,record_count,missing_values,minimum,first_quartile,median,mean,third_quartile,maximum
donor_age,"34,403",0,16,42.00,42.00,43.47,42.00,110


In [72]:
# Review most common donor ages
most_common_ages = (
    df[AGE_SOURCE_COLUMN]
    .value_counts()
    .head(15)
    .rename_axis(AGE_SOURCE_COLUMN)
    .reset_index(name="record_count")
)

most_common_ages["percentage"] = (
    most_common_ages["record_count"]
    .div(len(df))
    .mul(100)
)

display(most_common_ages.style
    .hide(axis="index")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "center")],
    }])
    .format({
        AGE_SOURCE_COLUMN: "{:,.0f}",
        "record_count": "{:,.0f}",
        "percentage": "{:,.2f}%",
    })
)

donor_age,record_count,percentage
42,"21,472",62.41%
32,384,1.12%
30,368,1.07%
31,351,1.02%
27,337,0.98%
33,332,0.97%
35,326,0.95%
29,323,0.94%
34,319,0.93%
37,308,0.90%


In [73]:
# Create age features
AGE_GROUP_BINS = [
    16,
    18,
    25,
    35,
    45,
    55,
    65,
    75,
    np.inf,
]

AGE_GROUP_LABELS = [
    "16–17",
    "18–24",
    "25–34",
    "35–44",
    "45–54",
    "55–64",
    "65–74",
    "75+",
]


df["feature_age_missing_flag"] = (
    df[AGE_SOURCE_COLUMN]
    .isna()
    .astype("int8")
)


df["feature_age_group"] = pd.cut(
    df[AGE_SOURCE_COLUMN],
    bins=AGE_GROUP_BINS,
    labels=AGE_GROUP_LABELS,
    right=False,
    include_lowest=True,
    ordered=True,
)


df["feature_age_decade"] = (
    np.floor(
        df[AGE_SOURCE_COLUMN] / 10
    )
    .mul(10)
    .astype("Int64")
)


df["feature_age_squared"] = (
    df[AGE_SOURCE_COLUMN]
    .astype("float64")
    .pow(2)
)

In [74]:
# Validate age features
AGE_FEATURES = [
    "feature_age_group",
    "feature_age_decade",
    "feature_age_squared",
    "feature_age_missing_flag",
]

age_feature_validation = pd.DataFrame(
    {
        "missing_values": (
            df[AGE_FEATURES]
            .isna()
            .sum()
        ),
        "unique_values": [
            df[column].nunique(dropna=True)
            for column in AGE_FEATURES
        ],
    }
)

display(age_feature_validation.style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "left")],
    }])
    .format("{:,.0f}")
)



assert set(
    df["feature_age_missing_flag"].unique()
).issubset({0, 1}), (
    "Unexpected values found in the age missing flag."
)

assert (
    df["feature_age_missing_flag"]
    == (
        df[AGE_SOURCE_COLUMN]
        .isna()
        .astype("int8")
    )
).all(), "Age missing flag validation failed."


assert (
    df.loc[
        df[AGE_SOURCE_COLUMN].notna(),
        "feature_age_group",
    ]
    .notna()
    .all()
), "At least one valid age was not assigned to an age group."

assert (
    df["feature_age_group"].isna()
    == df[AGE_SOURCE_COLUMN].isna()
).all(), (
    "Age group missingness does not match "
    "donor age missingness."
)


expected_age_decade = (
    np.floor(
        df[AGE_SOURCE_COLUMN] / 10
    )
    .mul(10)
    .astype("Int64")
)

assert df["feature_age_decade"].equals(
    expected_age_decade
), "Age decade validation failed."

assert (
    df["feature_age_decade"].isna()
    == df[AGE_SOURCE_COLUMN].isna()
).all(), (
    "Age decade missingness does not match "
    "donor age missingness."
)


expected_age_squared = (
    df[AGE_SOURCE_COLUMN]
    .astype("float64")
    .pow(2)
)

assert np.allclose(
    df["feature_age_squared"],
    expected_age_squared,
    equal_nan=True,
), "Squared age validation failed."

assert (
    df["feature_age_squared"].isna()
    == df[AGE_SOURCE_COLUMN].isna()
).all(), (
    "Squared age missingness does not match "
    "donor age missingness."
)

assert (
    df.loc[
        df["feature_age_squared"].notna(),
        "feature_age_squared",
    ]
    >= 0
).all(), "Negative squared age values were found."

print("Age feature validation passed.")

,missing_values,unique_values
feature_age_group,0,8
feature_age_decade,0,11
feature_age_squared,0,88
feature_age_missing_flag,0,1


Age feature validation passed.


In [75]:
# Summarize age group distribution
age_group_record_counts = (
    df["feature_age_group"]
    .value_counts(sort=False)
)

age_group_summary = pd.DataFrame(
    {
        "record_count": age_group_record_counts,
        "percentage": (
            age_group_record_counts
            .div(len(df))
            .mul(100)
        ),
    }
)

display(age_group_summary.style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "center")],
    }])
    .format({
        "record_count": "{:,.0f}",
        "percentage": "{:,.2f}%",
    })
)

,record_count,percentage
feature_age_group,,
16–17,52,0.15%
18–24,"1,100",3.20%
25–34,"3,278",9.53%
35–44,"23,863",69.36%
45–54,"2,115",6.15%
55–64,"1,680",4.88%
65–74,"1,263",3.67%
75+,"1,052",3.06%


In [76]:
# Summarize age decade distribution
age_decade_record_counts = (
    df["feature_age_decade"]
    .value_counts()
    .sort_index()
)

age_decade_summary = pd.DataFrame(
    {
        "record_count": age_decade_record_counts,
        "percentage": (
            age_decade_record_counts
            .div(len(df))
            .mul(100)
        ),
    }
)

age_decade_summary.index.name = (
    "feature_age_decade"
)

display(age_decade_summary.style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "center")],
    }])
    .format({
        "record_count": "{:,.0f}",
        "percentage": "{:,.2f}%",
    })
)

,record_count,percentage
feature_age_decade,,
10,170,0.49%
20,"2,506",7.28%
30,"3,248",9.44%
40,"23,488",68.27%
50,"1,890",5.49%
60,"1,408",4.09%
70,"1,021",2.97%
80,497,1.44%
90,169,0.49%


## Age Feature Engineering

The donor age distribution was reviewed before creating age-based features. The dataset contains 34,403 records with no missing age values. Ages range from 16 to 110, with a mean of 43.47 and a median of 42.

The distribution is heavily concentrated at age 42, which represents 21,472 records, or 62.41% of the dataset. The first quartile, median, and third quartile are also all 42. This unusual concentration should be considered when interpreting age-related model effects.

Because age may have a nonlinear relationship with donation likelihood, four features were created:
- `feature_age_group`
- `feature_age_decade`
- `feature_age_squared`
- `feature_age_missing_flag`

`feature_age_group` separates donors into eight interpretable age ranges:
- 16–17
- 18–24
- 25–34
- 35–44
- 45–54
- 55–64
- 65–74
- 75+

### Age Feature Details
- `feature_age_group` divides donors into eight age ranges from 16–17 through 75+. The 16–17 group was included because the cleaned dataset retains donors aged 16 and older. Records with `donor_age` below 16 were removed during data cleaning based on minimum Red Cross donation age requirements.
- The largest age group is 35–44, containing 23,863 records, or 69.36% of the dataset. This is primarily driven by the large concentration of records at age 42. Every other age group represents less than 10% of the dataset.
- `feature_age_decade` assigns each donor to the starting value of their age decade. For example, age 42 is represented as 40 and age 67 is represented as 60. The 40s decade contains 23,488 records, or 68.27% of the dataset.
- `feature_age_squared` was created to help models capture a nonlinear relationship between age and donation likelihood. This allows age-related effects to change across the age range rather than assuming a strictly linear pattern.
- `feature_age_missing_flag` identifies records with missing donor age values. Because no donor ages are missing, this feature is constant at 0 and may be removed during modeling because it has no variance.
- All four engineered age features passed validation. Every donor age was assigned to an age group and decade, the squared-age values were calculated correctly, and no unexpected missing values were introduced.

In [77]:
# Validate and summarize source features
AGE_SOURCE_COLUMN = "donor_age"
LOYALTY_SOURCE_COLUMN = "consecutive_donor_years"

CONSECUTIVE_DONOR_YEARS_PRE_CUTOFF_VERIFIED = False

required_age_loyalty_columns = [
    AGE_SOURCE_COLUMN,
    LOYALTY_SOURCE_COLUMN,
]

missing_required_columns = [
    column
    for column in required_age_loyalty_columns
    if column not in df.columns
]

assert not missing_required_columns, (
    "Required source columns are missing: "
    f"{missing_required_columns}"
)



for column in required_age_loyalty_columns:
    assert pd.api.types.is_numeric_dtype(
        df[column]
    ), f"{column} must be numeric."


assert (
    df.loc[
        df[AGE_SOURCE_COLUMN].notna(),
        AGE_SOURCE_COLUMN,
    ]
    >= 16
).all(), "Donor ages below 16 were found."


assert (
    df.loc[
        df[LOYALTY_SOURCE_COLUMN].notna(),
        LOYALTY_SOURCE_COLUMN,
    ]
    >= 0
).all(), "Negative consecutive donor years were found."


nonmissing_consecutive_years = (
    df[LOYALTY_SOURCE_COLUMN]
    .dropna()
    .to_numpy(dtype="float64")
)

assert np.allclose(
    nonmissing_consecutive_years,
    np.round(nonmissing_consecutive_years),
), "Noninteger consecutive donor year values were found."


age_loyalty_source_summary = pd.DataFrame(
    {
        "record_count": [
            df[AGE_SOURCE_COLUMN].count(),
            df[LOYALTY_SOURCE_COLUMN].count(),
        ],
        "missing_values": [
            df[AGE_SOURCE_COLUMN].isna().sum(),
            df[LOYALTY_SOURCE_COLUMN].isna().sum(),
        ],
        "minimum": [
            df[AGE_SOURCE_COLUMN].min(),
            df[LOYALTY_SOURCE_COLUMN].min(),
        ],
        "median": [
            df[AGE_SOURCE_COLUMN].median(),
            df[LOYALTY_SOURCE_COLUMN].median(),
        ],
        "mean": [
            df[AGE_SOURCE_COLUMN].mean(),
            df[LOYALTY_SOURCE_COLUMN].mean(),
        ],
        "maximum": [
            df[AGE_SOURCE_COLUMN].max(),
            df[LOYALTY_SOURCE_COLUMN].max(),
        ],
    },
    index=required_age_loyalty_columns,
)

display(age_loyalty_source_summary.style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "center")],
    }])
    .format({
        "record_count": "{:,.0f}",
        "missing_values": "{:,.0f}",
        "minimum": "{:,.2f}",
        "median": "{:,.2f}",
        "mean": "{:,.2f}",
        "maximum": "{:,.2f}",
    })
)

,record_count,missing_values,minimum,median,mean,maximum
donor_age,"34,403",0,16.00,42.00,43.47,110.00
consecutive_donor_years,"34,403",0,0.00,0.00,1.14,36.00


In [78]:
# Create experimental age loyalty feature
AGE_ADJUSTED_LOYALTY_FEATURE = (
    "feature_donor_years_age_ratio"
)


df[AGE_ADJUSTED_LOYALTY_FEATURE] = (
    df[LOYALTY_SOURCE_COLUMN]
    .astype("float64")
    .div(
        df[AGE_SOURCE_COLUMN]
        .astype("float64")
    )
)

In [79]:
# Validate and summarize experimental feature
expected_donor_years_age_ratio = (
    df[LOYALTY_SOURCE_COLUMN]
    .astype("float64")
    .div(
        df[AGE_SOURCE_COLUMN]
        .astype("float64")
    )
)


assert np.allclose(
    df[AGE_ADJUSTED_LOYALTY_FEATURE],
    expected_donor_years_age_ratio,
    equal_nan=True,
), "Age adjusted loyalty ratio validation failed."



expected_missing_mask = (
    df[AGE_SOURCE_COLUMN].isna()
    | df[LOYALTY_SOURCE_COLUMN].isna()
)

assert (
    df[AGE_ADJUSTED_LOYALTY_FEATURE].isna()
    == expected_missing_mask
).all(), (
    "Feature missingness does not match "
    "the source column missingness."
)



nonmissing_ratio_values = (
    df[AGE_ADJUSTED_LOYALTY_FEATURE]
    .dropna()
    .to_numpy(dtype="float64")
)

assert np.isfinite(
    nonmissing_ratio_values
).all(), "Infinite ratio values were found."



assert (
    df.loc[
        df[AGE_ADJUSTED_LOYALTY_FEATURE].notna(),
        AGE_ADJUSTED_LOYALTY_FEATURE,
    ]
    >= 0
).all(), "Negative ratio values were found."



age_adjusted_loyalty_summary = pd.DataFrame(
    {
        "record_count": [
            df[AGE_ADJUSTED_LOYALTY_FEATURE].count()
        ],
        "missing_values": [
            df[AGE_ADJUSTED_LOYALTY_FEATURE].isna().sum()
        ],
        "unique_values": [
            df[AGE_ADJUSTED_LOYALTY_FEATURE].nunique()
        ],
        "minimum": [
            df[AGE_ADJUSTED_LOYALTY_FEATURE].min()
        ],
        "first_quartile": [
            df[AGE_ADJUSTED_LOYALTY_FEATURE].quantile(0.25)
        ],
        "median": [
            df[AGE_ADJUSTED_LOYALTY_FEATURE].median()
        ],
        "mean": [
            df[AGE_ADJUSTED_LOYALTY_FEATURE].mean()
        ],
        "third_quartile": [
            df[AGE_ADJUSTED_LOYALTY_FEATURE].quantile(0.75)
        ],
        "maximum": [
            df[AGE_ADJUSTED_LOYALTY_FEATURE].max()
        ],
        "ratios_above_one": [
            (
                df[AGE_ADJUSTED_LOYALTY_FEATURE]
                > 1
            ).sum()
        ],
    },
    index=[AGE_ADJUSTED_LOYALTY_FEATURE],
)

display(age_adjusted_loyalty_summary.style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "center")],
    }])
    .format({
        "record_count": "{:,.0f}",
        "missing_values": "{:,.0f}",
        "unique_values": "{:,.0f}",
        "minimum": "{:,.4f}",
        "first_quartile": "{:,.4f}",
        "median": "{:,.4f}",
        "mean": "{:,.4f}",
        "third_quartile": "{:,.4f}",
        "maximum": "{:,.4f}",
        "ratios_above_one": "{:,.0f}",
    })
)

print(
    "\nExperimental age adjusted loyalty "
    "feature validation passed."
)

,record_count,missing_values,unique_values,minimum,first_quartile,median,mean,third_quartile,maximum,ratios_above_one
feature_donor_years_age_ratio,"34,403",0,423,0.0000,0.0000,0.0000,0.0266,0.0238,1.0000,0



Experimental age adjusted loyalty feature validation passed.


In [80]:
# Compare experimental feature across age groups
assert "feature_age_group" in df.columns, (
    "feature_age_group must be created before "
    "reviewing the age adjusted loyalty feature."
)

age_adjusted_loyalty_by_group = (
    df.groupby(
        "feature_age_group",
        observed=False,
    )
    .agg(
        record_count=(
            AGE_ADJUSTED_LOYALTY_FEATURE,
            "size",
        ),
        median_donor_age=(
            AGE_SOURCE_COLUMN,
            "median",
        ),
        mean_consecutive_donor_years=(
            LOYALTY_SOURCE_COLUMN,
            "mean",
        ),
        median_consecutive_donor_years=(
            LOYALTY_SOURCE_COLUMN,
            "median",
        ),
        mean_donor_years_age_ratio=(
            AGE_ADJUSTED_LOYALTY_FEATURE,
            "mean",
        ),
        median_donor_years_age_ratio=(
            AGE_ADJUSTED_LOYALTY_FEATURE,
            "median",
        ),
    )
)

display(age_adjusted_loyalty_by_group.style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "center")],
    }])
    .format({
        "record_count": "{:,.0f}",
        "median_donor_age": "{:,.2f}",
        "mean_consecutive_donor_years": "{:,.2f}",
        "median_consecutive_donor_years": "{:,.2f}",
        "mean_donor_years_age_ratio": "{:,.4f}",
        "median_donor_years_age_ratio": "{:,.4f}",
    })
)

,record_count,median_donor_age,mean_consecutive_donor_years,median_consecutive_donor_years,mean_donor_years_age_ratio,median_donor_years_age_ratio
feature_age_group,,,,,,
16–17,52,17.00,0.08,0.00,0.0045,0.0000
18–24,"1,100",23.00,0.27,0.00,0.0123,0.0000
25–34,"3,278",30.00,0.46,0.00,0.0159,0.0000
35–44,"23,863",42.00,1.33,1.00,0.0318,0.0238
45–54,"2,115",49.00,0.68,0.00,0.0139,0.0000
55–64,"1,680",59.00,0.83,0.00,0.0142,0.0000
65–74,"1,263",70.00,1.03,0.00,0.0148,0.0000
75+,"1,052",82.00,1.42,0.00,0.0170,0.0000


## Experimental Age Adjusted Loyalty Feature

An experimental age adjusted loyalty feature was created using:

```python
feature_donor_years_age_ratio = consecutive_donor_years / donor_age
```

This feature places consecutive donor behavior in the context of the donor’s age. It is intended to supplement, rather than replace, `donor_age`, `consecutive_donor_years`, and `feature_age_group`.

### Source Feature Summary
- Both `donor_age` and `consecutive_donor_years` contain 34,403 nonmissing values.
- Donor age ranges from 16 to 110, with a median of 42.
- Consecutive donor years range from 0 to 36, with a median of 0 and a mean of 1.14.
- The zero median indicates that at least half of the records have no consecutive donor years.

### Experimental Feature Results
- `feature_donor_years_age_ratio` contains no missing or infinite values.
- The feature has 423 unique values.
- The median ratio is 0 because most records have no consecutive donor years.
- The mean ratio is 0.0266, and the third quartile is 0.0238.
- The maximum ratio is 1.0000.
- No ratios exceed 1, indicating that no donor has more consecutive donor years than their recorded age.

### Comparison Across Age Groups
- The 35–44 group has the highest mean ratio at 0.0318 and is the only group with a nonzero median ratio of 0.0238.
- This result is influenced by the large concentration of records at age 42 and the group’s median of one consecutive donor year.
- The 75+ group has the highest mean number of consecutive donor years at 1.42, but its mean ratio is lower at 0.0170 because donor age is included in the denominator.
- Younger groups have lower average consecutive donor years and correspondingly low age adjusted ratios.
- The comparison shows that donors with similar consecutive donation histories may receive different ratio values depending on their age.

### Feature Status

All mathematical validations passed. However, this ratio should not be interpreted as the exact proportion of a donor’s eligible adult life spent donating because the denominator is total age rather than years since donation eligibility.

The timing of `consecutive_donor_years` has not yet been confirmed. Therefore, `feature_donor_years_age_ratio` remains experimental and should not be included in Phase 5 modeling until `consecutive_donor_years` is verified as available before the prediction cutoff.


In [81]:
# Define categorical columns
GENDER_SOURCE_COLUMN = "gender_identity"
ADDRESS_SOURCE_COLUMN = "preferred_address_type"
POSTAL_SOURCE_COLUMN = "donor_postal_code"

CATEGORICAL_SOURCE_COLUMNS = [
    GENDER_SOURCE_COLUMN,
    ADDRESS_SOURCE_COLUMN,
    POSTAL_SOURCE_COLUMN,
]

missing_categorical_columns = [
    column
    for column in CATEGORICAL_SOURCE_COLUMNS
    if column not in df.columns
]

assert not missing_categorical_columns, (
    "Required categorical columns are missing: "
    f"{missing_categorical_columns}"
)



RARE_CATEGORY_MIN_COUNT = max(
    50,
    int(np.ceil(len(df) * 0.0025)),
)

print(
    "Rare-category minimum count:",
    f"{RARE_CATEGORY_MIN_COUNT:,}",
)

Rare-category minimum count: 87


In [82]:
# Summarize source categorical features
categorical_source_summary = pd.DataFrame(
    {
        "dtype": [
            str(df[column].dtype)
            for column in CATEGORICAL_SOURCE_COLUMNS
        ],
        "record_count": [
            df[column].count()
            for column in CATEGORICAL_SOURCE_COLUMNS
        ],
        "missing_values": [
            df[column].isna().sum()
            for column in CATEGORICAL_SOURCE_COLUMNS
        ],
        "unique_values": [
            df[column].nunique(dropna=True)
            for column in CATEGORICAL_SOURCE_COLUMNS
        ],
    },
    index=CATEGORICAL_SOURCE_COLUMNS,
)

display(categorical_source_summary.style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "left")],
    }])
    .format({
        "record_count": "{:,.0f}",
        "missing_values": "{:,.0f}",
        "unique_values": "{:,.0f}",
    })
)

,dtype,record_count,missing_values,unique_values
gender_identity,str,"33,912",491,3
preferred_address_type,str,"30,370","4,033",4
donor_postal_code,str,"34,312",91,"20,944"


In [83]:
# Summarize gender categories
gender_source_display = (
    df[GENDER_SOURCE_COLUMN]
    .astype("string")
    .str.strip()
    .replace(r"^\s*$", pd.NA, regex=True)
    .fillna("Missing")
)

gender_source_counts = (
    gender_source_display
    .value_counts(dropna=False)
    .rename_axis(GENDER_SOURCE_COLUMN)
    .reset_index(name="record_count")
)

gender_source_counts["percentage"] = (
    gender_source_counts["record_count"]
    .div(len(df))
    .mul(100)
)

display(gender_source_counts.style
    .hide(axis="index")
    .format({
        "record_count": "{:,.0f}",
        "percentage": "{:,.2f}%",
    })
)

gender_identity,record_count,percentage
Female,"16,631",48.34%
Male,"16,178",47.02%
Unknown,"1,103",3.21%
Missing,491,1.43%


In [84]:
# Summarize address categories 
address_source_display = (
    df[ADDRESS_SOURCE_COLUMN]
    .astype("string")
    .str.strip()
    .replace(r"^\s*$", pd.NA, regex=True)
    .fillna("Missing")
)

address_source_counts = (
    address_source_display
    .value_counts(dropna=False)
    .rename_axis(ADDRESS_SOURCE_COLUMN)
    .reset_index(name="record_count")
)

address_source_counts["percentage"] = (
    address_source_counts["record_count"]
    .div(len(df))
    .mul(100)
)

display(address_source_counts.style
    .hide(axis="index")
    .format({
        "record_count": "{:,.0f}",
        "percentage": "{:,.2f}%",
    })
)

preferred_address_type,record_count,percentage
Home,"28,677",83.36%
Missing,"4,033",11.72%
Business,975,2.83%
Campus,647,1.88%
Other,71,0.21%


In [85]:
# Group infrequent categorical values into other
def group_rare_categories(
    series,
    minimum_count,
    protected_values=None,
):
    protected_values = set(
        protected_values or []
    )

    category_counts = series.value_counts(
        dropna=False
    )

    rare_categories = [
        category
        for category, count in category_counts.items()
        if (
            count < minimum_count
            and category not in protected_values
        )
    ]

    return series.where(
        ~series.isin(rare_categories),
        "Other",
    )

In [86]:
# Standardize gender values
gender_normalized = (
    df[GENDER_SOURCE_COLUMN]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
)


gender_missing_mask = (
    df[GENDER_SOURCE_COLUMN].isna()
    | gender_normalized.eq("")
)


GENDER_STANDARDIZATION_MAP = {
    "female": "Female",
    "f": "Female",
    "male": "Male",
    "m": "Male",
    "nonbinary": "Nonbinary",
    "non-binary": "Nonbinary",
    "non binary": "Nonbinary",
    "unknown": "Unknown",
    "uknown": "Unknown",
    "u": "Unknown",
    "unk": "Unknown",
    "unspecified": "Unknown",
    "not specified": "Unknown",
    "prefer not to say": "Unknown",
    "": "Unknown",
}

gender_clean = gender_normalized.map(
    GENDER_STANDARDIZATION_MAP
)


unmapped_gender_mask = (
    gender_clean.isna()
    & gender_normalized.notna()
)

gender_clean.loc[unmapped_gender_mask] = (
    gender_normalized.loc[
        unmapped_gender_mask
    ]
    .str.title()
)

gender_clean = gender_clean.fillna(
    "Unknown"
)


df["feature_gender_identity"] = (
    group_rare_categories(
        gender_clean,
        minimum_count=RARE_CATEGORY_MIN_COUNT,
        protected_values={
            "Female",
            "Male",
            "Nonbinary",
            "Unknown",
            "Other",
        },
    )
    .astype("string")
)


df["feature_gender_missing_flag"] = (
    gender_missing_mask
    .astype("int8")
)


gender_feature_summary = (
    df["feature_gender_identity"]
    .value_counts(dropna=False)
    .rename_axis("feature_gender_identity")
    .reset_index(name="record_count")
)

gender_feature_summary["percentage"] = (
    gender_feature_summary["record_count"]
    .div(len(df))
    .mul(100)
)

display(gender_feature_summary.style
    .hide(axis="index")
    .format({
        "record_count": "{:,.0f}",
        "percentage": "{:,.2f}%",
    })
)

feature_gender_identity,record_count,percentage
Female,"16,631",48.34%
Male,"16,178",47.02%
Unknown,"1,594",4.63%


In [87]:
# Standardize preferred address type
address_normalized = (
    df[ADDRESS_SOURCE_COLUMN]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
)

ADDRESS_MISSING_TOKENS = {
    "",
    "unknown",
    "u",
    "none",
    "nan",
    "not provided",
    "not specified",
    "unspecified",
}

address_missing_mask = (
    df[ADDRESS_SOURCE_COLUMN].isna()
    | address_normalized.isin(
        ADDRESS_MISSING_TOKENS
    )
)


ADDRESS_STANDARDIZATION_MAP = {
    "home": "Home",
    "home address": "Home",
    "residence": "Home",
    "residential": "Home",
    "business": "Business",
    "business address": "Business",
    "work": "Business",
    "work address": "Business",
    "office": "Business",
    "campus": "Campus",
    "campus address": "Campus",
    "seasonal": "Seasonal",
    "temporary": "Seasonal",
    "other": "Other",
}

address_clean = address_normalized.map(
    ADDRESS_STANDARDIZATION_MAP
)


unmapped_address_mask = (
    address_clean.isna()
    & ~address_missing_mask
)

address_clean.loc[unmapped_address_mask] = (
    address_normalized.loc[
        unmapped_address_mask
    ]
    .str.title()
)

address_clean.loc[address_missing_mask] = (
    "Missing"
)



df["feature_preferred_address_type"] = (
    group_rare_categories(
        address_clean,
        minimum_count=RARE_CATEGORY_MIN_COUNT,
        protected_values={
            "Home",
            "Business",
            "Campus",
            "Seasonal",
            "Missing",
            "Other",
        },
    )
    .astype("string")
)



df[
    "feature_preferred_address_missing_flag"
] = (
    address_missing_mask
    .astype("int8")
)


address_feature_summary = (
    df["feature_preferred_address_type"]
    .value_counts(dropna=False)
    .rename_axis(
        "feature_preferred_address_type"
    )
    .reset_index(name="record_count")
)

address_feature_summary["percentage"] = (
    address_feature_summary["record_count"]
    .div(len(df))
    .mul(100)
)

display(address_feature_summary.style
    .hide(axis="index")
    .format({
        "record_count": "{:,.0f}",
        "percentage": "{:,.2f}%",
    })
)

feature_preferred_address_type,record_count,percentage
Home,"28,677",83.36%
Missing,"4,033",11.72%
Business,975,2.83%
Campus,647,1.88%
Other,71,0.21%


In [88]:
# Standardize postal codes
if not (
    pd.api.types.is_string_dtype(
        df[POSTAL_SOURCE_COLUMN]
    )
    or isinstance(
        df[POSTAL_SOURCE_COLUMN].dtype,
        pd.StringDtype,
    )
):
    warnings.warn(
        "donor_postal_code was not loaded as a "
        "string. Leading zeros may already have "
        "been lost. Reload the CSV with "
        "dtype={'donor_postal_code': 'string'} "
        "before finalizing this feature."
    )


postal_code_clean = (
    df[POSTAL_SOURCE_COLUMN]
    .astype("string")
    .str.strip()
    .str.upper()
)



postal_code_clean = (
    postal_code_clean
    .str.replace(
        r"\.0$",
        "",
        regex=True,
    )
)


POSTAL_MISSING_TOKENS = {
    "",
    "NAN",
    "NONE",
    "UNKNOWN",
    "UNSPECIFIED",
    "NOT PROVIDED",
}

postal_missing_mask = (
    df[POSTAL_SOURCE_COLUMN].isna()
    | postal_code_clean.isin(
        POSTAL_MISSING_TOKENS
    )
)

postal_code_clean = postal_code_clean.mask(
    postal_missing_mask,
    pd.NA,
)


short_numeric_postal_mask = (
    postal_code_clean
    .str.fullmatch(
        r"\d{1,5}",
        na=False,
    )
)

postal_code_clean.loc[
    short_numeric_postal_mask
] = (
    postal_code_clean.loc[
        short_numeric_postal_mask
    ]
    .str.zfill(5)
)



df["feature_donor_postal_code"] = (
    postal_code_clean
    .astype("string")
)



df["feature_postal_code_missing_flag"] = (
    postal_missing_mask
    .astype("int8")
)

In [89]:
# Postal code cardinality
postal_code_counts = (
    df["feature_donor_postal_code"]
    .value_counts(dropna=True)
)

postal_singleton_levels = (
    postal_code_counts == 1
).sum()

postal_singleton_records = (
    postal_code_counts[
        postal_code_counts == 1
    ]
    .sum()
)

postal_cardinality_summary = pd.DataFrame(
    {
        "record_count": [
            df["feature_donor_postal_code"]
            .count()
        ],
        "missing_values": [
            df["feature_donor_postal_code"]
            .isna()
            .sum()
        ],
        "unique_values": [
            df["feature_donor_postal_code"]
            .nunique(dropna=True)
        ],
        "cardinality_percentage": [
            (
                df["feature_donor_postal_code"]
                .nunique(dropna=True)
                / len(df)
                * 100
            )
        ],
        "singleton_levels": [
            postal_singleton_levels
        ],
        "records_in_singleton_levels": [
            postal_singleton_records
        ],
    },
    index=["feature_donor_postal_code"],
)

display(postal_cardinality_summary.style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "center")],
    }])
    .format({
        "record_count": "{:,.0f}",
        "missing_values": "{:,.0f}",
        "unique_values": "{:,.0f}",
        "cardinality_percentage": "{:,.2f}%",
        "singleton_levels": "{:,.0f}",
        "records_in_singleton_levels": "{:,.0f}",
    })
)

,record_count,missing_values,unique_values,cardinality_percentage,singleton_levels,records_in_singleton_levels
feature_donor_postal_code,"34,312",91,"20,944",60.88%,"14,868","14,868"


In [90]:
# 3 digit postal prefix
df["feature_postal_prefix_3"] = (
    df["feature_donor_postal_code"]
    .str.extract(
        r"^(\d{3})",
        expand=False,
    )
    .astype("string")
)


postal_prefix_for_grouping = (
    df["feature_postal_prefix_3"]
    .fillna("Missing")
)

df["feature_postal_prefix_3_grouped"] = (
    group_rare_categories(
        postal_prefix_for_grouping,
        minimum_count=RARE_CATEGORY_MIN_COUNT,
        protected_values={
            "Missing",
            "Other",
        },
    )
    .astype("string")
)

postal_feature_comparison = pd.DataFrame(
    {
        "missing_values": [
            df["feature_donor_postal_code"]
            .isna()
            .sum(),
            df["feature_postal_prefix_3"]
            .isna()
            .sum(),
            df[
                "feature_postal_prefix_3_grouped"
            ]
            .isna()
            .sum(),
        ],
        "unique_values": [
            df["feature_donor_postal_code"]
            .nunique(dropna=True),
            df["feature_postal_prefix_3"]
            .nunique(dropna=True),
            df[
                "feature_postal_prefix_3_grouped"
            ]
            .nunique(dropna=True),
        ],
    },
    index=[
        "feature_donor_postal_code",
        "feature_postal_prefix_3",
        "feature_postal_prefix_3_grouped",
    ],
)

display(postal_feature_comparison.style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "left")],
    }])
    .format("{:,.0f}")
)

,missing_values,unique_values
feature_donor_postal_code,91,"20,944"
feature_postal_prefix_3,91,941
feature_postal_prefix_3_grouped,0,3


In [91]:
# Derive the first three digits of valid numeric postal codes
df["feature_postal_prefix_3"] = (
    df["feature_donor_postal_code"]
    .str.extract(
        r"^(\d{3})",
        expand=False,
    )
    .astype("string")
)


# Compare full postal-code and prefix cardinality
postal_feature_comparison = pd.DataFrame(
    {
        "missing_values": [
            df["feature_donor_postal_code"]
            .isna()
            .sum(),
            df["feature_postal_prefix_3"]
            .isna()
            .sum(),
        ],
        "unique_values": [
            df["feature_donor_postal_code"]
            .nunique(dropna=True),
            df["feature_postal_prefix_3"]
            .nunique(dropna=True),
        ],
        "cardinality_percentage": [
            (
                df["feature_donor_postal_code"]
                .nunique(dropna=True)
                / len(df)
                * 100
            ),
            (
                df["feature_postal_prefix_3"]
                .nunique(dropna=True)
                / len(df)
                * 100
            ),
        ],
    },
    index=[
        "feature_donor_postal_code",
        "feature_postal_prefix_3",
    ],
)

display(postal_feature_comparison.style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "left")],
    }])
    .format({
        "missing_values": "{:,.0f}",
        "unique_values": "{:,.0f}",
        "cardinality_percentage": "{:,.2f}%",
    })
)

,missing_values,unique_values,cardinality_percentage
feature_donor_postal_code,91,"20,944",60.88%
feature_postal_prefix_3,91,941,2.74%


In [92]:
# Review the most common three-digit prefixes
postal_prefix_summary = (
    df["feature_postal_prefix_3"]
    .fillna("Missing")
    .value_counts(dropna=False)
    .rename_axis("feature_postal_prefix_3")
    .reset_index(name="record_count")
)

postal_prefix_summary["percentage"] = (
    postal_prefix_summary["record_count"]
    .div(len(df))
    .mul(100)
)

display(postal_prefix_summary.head(20).style
    .hide(axis="index")
    .format({
        "record_count": "{:,.0f}",
        "percentage": "{:,.2f}%",
    })
)

feature_postal_prefix_3,record_count,percentage
902,"5,864",17.05%
Missing,91,0.26%
606,77,0.22%
900,71,0.21%
770,70,0.20%
317,69,0.20%
996,67,0.19%
956,67,0.19%
530,66,0.19%
790,66,0.19%


In [93]:
# Validate categorical features
ENGINEERED_CATEGORICAL_FEATURES = [
    "feature_gender_identity",
    "feature_gender_missing_flag",
    "feature_preferred_address_type",
    "feature_preferred_address_missing_flag",
    "feature_donor_postal_code",
    "feature_postal_code_missing_flag",
    "feature_postal_prefix_3",
    "feature_postal_prefix_3_grouped",
]


CATEGORICAL_MISSING_FLAGS = [
    "feature_gender_missing_flag",
    "feature_preferred_address_missing_flag",
    "feature_postal_code_missing_flag",
]

for column in CATEGORICAL_MISSING_FLAGS:
    assert set(
        df[column].unique()
    ).issubset({0, 1}), (
        f"Unexpected values found in {column}."
    )


assert (
    df["feature_gender_identity"]
    .notna()
    .all()
), "Missing standardized gender values were found."

assert (
    df["feature_preferred_address_type"]
    .notna()
    .all()
), (
    "Missing standardized preferred address "
    "values were found."
)

assert (
    pd.api.types.is_string_dtype(
        df["feature_donor_postal_code"]
    )
), "Postal codes were not preserved as strings."

five_digit_postal_mask = (
    df["feature_donor_postal_code"]
    .str.fullmatch(
        r"\d{5}",
        na=False,
    )
)

assert (
    df.loc[
        five_digit_postal_mask,
        "feature_donor_postal_code",
    ]
    .str.len()
    .eq(5)
    .all()
), "Invalid five-digit postal-code formatting found."


categorical_feature_validation = pd.DataFrame(
    {
        "missing_values": [
            df[column].isna().sum()
            for column
            in ENGINEERED_CATEGORICAL_FEATURES
        ],
        "unique_values": [
            df[column].nunique(dropna=True)
            for column
            in ENGINEERED_CATEGORICAL_FEATURES
        ],
    },
    index=ENGINEERED_CATEGORICAL_FEATURES,
)

display(categorical_feature_validation.style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "left")],
    }])
    .format("{:,.0f}")
)

print(
    "\nCategorical feature validation passed."
)

,missing_values,unique_values
feature_gender_identity,0,3
feature_gender_missing_flag,0,2
feature_preferred_address_type,0,5
feature_preferred_address_missing_flag,0,2
feature_donor_postal_code,91,"20,944"
feature_postal_code_missing_flag,0,2
feature_postal_prefix_3,91,941
feature_postal_prefix_3_grouped,0,3



Categorical feature validation passed.


## Categorical Feature Preparation

The categorical features `gender_identity`, `preferred_address_type`, and `donor_postal_code` were reviewed and standardized before modeling. A rare-category threshold of 87 records (approximately 0.25% of the dataset) was used when evaluating infrequent categories.

### Gender Identity

- The original feature contained 491 missing values and three nonmissing categories.
- Unknown and missing values were consolidated into `Unknown`.
- The standardized `feature_gender_identity` contains:

  - Female: 16,631 records (48.34%)
  - Male: 16,178 records (47.02%)
  - Unknown: 1,594 records (4.63%)

- `feature_gender_missing_flag` preserves whether the original value was missing before consolidation.

### Preferred Address Type

- The original feature contained 4,033 missing values.
- The standardized categories are:

  - Home: 28,677 records (83.36%)
  - Missing: 4,033 records (11.72%)
  - Business: 975 records (2.83%)
  - Campus: 647 records (1.88%)
  - Other: 71 records (0.21%)
- Missing values were represented explicitly as `Missing`.
- `feature_preferred_address_missing_flag` preserves the original missingness.

### Postal Code Preparation

- Postal codes were preserved as strings to retain leading zeros.
- The standardized postal-code feature contains 34,312 nonmissing values, 91 missing values, and 20,944 unique values.
- Postal codes have a cardinality of 60.88%, with 14,868 codes appearing only once, making the full postal code too granular for the baseline model.

A three-digit postal prefix was also derived:

- `feature_postal_prefix_3` contains 941 unique prefixes with a cardinality of 2.74%.
- The most common prefix is `902`, representing 5,864 records (17.05%).
- Most remaining prefixes each represent less than 0.25% of the dataset.

The grouped postal-prefix feature was discarded because the rare-category threshold reduced it to only `902`, `Other`, and `Missing`, making it primarily an indicator for prefix `902` rather than a useful geographic feature.

### Modeling Considerations

* `feature_gender_identity` and `feature_preferred_address_type` can be included in the baseline model after categorical encoding.
- The missing-value indicators can be evaluated separately for predictive value.
- The full postal code should be excluded from the baseline model because of its high cardinality.
- The three-digit postal prefix should be evaluated only in a separate geography comparison model.
- Any grouping or encoding of postal prefixes should be learned from the training data to avoid data leakage.

### Privacy and Fairness Considerations
- Postal-code features may indirectly represent socioeconomic or demographic differences, so any geography comparison model should be reviewed for privacy and fairness concerns before the feature is retained.




All engineered categorical features passed validation. Standardized gender and address categories contain no missing values, missing indicators are binary, and postal codes remain stored as strings.


In [94]:
# Validate address types
ADDRESS_TYPE_SOURCE_COLUMN = (
    "feature_preferred_address_type"
)

ADDRESS_TYPE_MISSING_FLAG = (
    "feature_preferred_address_missing_flag"
)

EXPECTED_ADDRESS_TYPES = {
    "Home",
    "Business",
    "Campus",
    "Other",
    "Missing",
}


required_address_columns = [
    ADDRESS_TYPE_SOURCE_COLUMN,
    ADDRESS_TYPE_MISSING_FLAG,
]

missing_address_columns = [
    column
    for column in required_address_columns
    if column not in df.columns
]

assert not missing_address_columns, (
    "Required address-type columns are missing: "
    f"{missing_address_columns}"
)


observed_address_types = set(
    df[ADDRESS_TYPE_SOURCE_COLUMN]
    .dropna()
    .unique()
)

unexpected_address_types = (
    observed_address_types
    - EXPECTED_ADDRESS_TYPES
)

assert not unexpected_address_types, (
    "Unexpected address-type categories found: "
    f"{unexpected_address_types}"
)

assert (
    df[ADDRESS_TYPE_SOURCE_COLUMN]
    .notna()
    .all()
), "Missing standardized address-type values found."



assert set(
    df[ADDRESS_TYPE_MISSING_FLAG].unique()
).issubset({0, 1}), (
    "Unexpected values found in the existing "
    "address-type missing flag."
)

assert (
    df[ADDRESS_TYPE_MISSING_FLAG]
    .eq(
        df[ADDRESS_TYPE_SOURCE_COLUMN]
        .eq("Missing")
        .astype("int8")
    )
    .all()
), (
    "The existing address-type missing flag "
    "does not match the standardized category."
)

In [95]:
# Create address type indicators
ADDRESS_TYPE_INDICATOR_MAP = {
    "Business": "feature_is_business_address",
    "Home": "feature_is_home_address",
    "Campus": "feature_is_campus_address",
    "Other": "feature_address_type_other",
}
for category, feature_name in (
    ADDRESS_TYPE_INDICATOR_MAP.items()
):
    df[feature_name] = (
        df[ADDRESS_TYPE_SOURCE_COLUMN]
        .eq(category)
        .astype("int8")
    )


ADDRESS_TYPE_INDICATOR_FEATURES = [
    *ADDRESS_TYPE_INDICATOR_MAP.values(),
    ADDRESS_TYPE_MISSING_FLAG,
]

In [96]:
# Validate and summarize indicators
for feature_name in (
    ADDRESS_TYPE_INDICATOR_FEATURES
):
    assert set(
        df[feature_name].unique()
    ).issubset({0, 1}), (
        f"Unexpected values found in "
        f"{feature_name}."
    )

for category, feature_name in (
    ADDRESS_TYPE_INDICATOR_MAP.items()
):
    expected_indicator = (
        df[ADDRESS_TYPE_SOURCE_COLUMN]
        .eq(category)
        .astype("int8")
    )

    assert (
        df[feature_name]
        .eq(expected_indicator)
        .all()
    ), (
        f"{feature_name} does not match "
        f"the {category} category."
    )


address_indicator_total = (
    df[ADDRESS_TYPE_INDICATOR_FEATURES]
    .sum(axis=1)
)

assert (
    address_indicator_total == 1
).all(), (
    "Each record should activate exactly one "
    "address-type indicator."
)

address_type_indicator_summary = (
    pd.DataFrame({
        "source_category": [
            *ADDRESS_TYPE_INDICATOR_MAP.keys(),
            "Missing",
        ],
        "feature_name": (
            ADDRESS_TYPE_INDICATOR_FEATURES
        ),
    })
)

address_type_indicator_summary[
    "record_count"
] = (
    address_type_indicator_summary[
        "feature_name"
    ]
    .map(
        lambda column: df[column].sum()
    )
)

address_type_indicator_summary[
    "percentage"
] = (
    address_type_indicator_summary[
        "record_count"
    ]
    .div(len(df))
    .mul(100)
)

display(address_type_indicator_summary.style
    .hide(axis="index")
    .set_properties(**{"text-align": "left"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "left")],
    }])
    .format({
        "record_count": "{:,.0f}",
        "percentage": "{:,.2f}%",
    })
)

print(
    "\nPreferred address-type indicator "
    "validation passed."
)

source_category,feature_name,record_count,percentage
Business,feature_is_business_address,975,2.83%
Home,feature_is_home_address,"28,677",83.36%
Campus,feature_is_campus_address,647,1.88%
Other,feature_address_type_other,71,0.21%
Missing,feature_preferred_address_missing_flag,"4,033",11.72%



Preferred address-type indicator validation passed.


## Preferred Address-Type Indicators

The standardized preferred address categories were converted into mutually exclusive binary indicators:

- `feature_is_business_address`: 975 records (2.83%)
- `feature_is_home_address`: 28,677 records (83.36%)
- `feature_is_campus_address`: 647 records (1.88%)
- `feature_address_type_other`: 71 records (0.21%)
- `feature_preferred_address_missing_flag`: 4,033 records (11.72%)

Each record activates exactly one address-type indicator, and all features passed binary and category-alignment validation.

`feature_is_business_address` only identifies a preferred business address. It does not indicate corporate giving or matching-gift eligibility.

Because Home represents 83.36% of the dataset and Other represents only 0.21%, Phase 5 should evaluate whether the less common indicators provide useful predictive value. Models should use either these indicators or model-specific encoding of `feature_preferred_address_type`, rather than automatically including both representations.


In [97]:
# Validate missingness inputs
GENDER_FEATURE_COLUMN = "feature_gender_identity"
GENDER_MISSING_FLAG = "feature_gender_missing_flag"
ADDRESS_MISSING_FLAG = (
    "feature_preferred_address_missing_flag"
)
POSTAL_CODE_FEATURE = "feature_donor_postal_code"
POSTAL_CODE_MISSING_FLAG = (
    "feature_postal_code_missing_flag"
)
AGE_SOURCE_COLUMN = "donor_age"
AGE_MISSING_FLAG = "feature_age_missing_flag"

required_missingness_columns = [
    GENDER_FEATURE_COLUMN,
    GENDER_MISSING_FLAG,
    ADDRESS_MISSING_FLAG,
    POSTAL_CODE_FEATURE,
    POSTAL_CODE_MISSING_FLAG,
    AGE_SOURCE_COLUMN,
    AGE_MISSING_FLAG,
]

missing_required_columns = [
    column
    for column in required_missingness_columns
    if column not in df.columns
]

assert not missing_required_columns, (
    "Required missingness columns are missing: "
    f"{missing_required_columns}"
)

In [98]:
# Create gender unknown indicator
df["feature_gender_unknown_flag"] = (
    df[GENDER_FEATURE_COLUMN]
    .eq("Unknown")
    .astype("int8")
)

In [99]:
# Validate missingness features
SELECTED_MISSINGNESS_FEATURES = [
    "feature_gender_unknown_flag",
    GENDER_MISSING_FLAG,
    ADDRESS_MISSING_FLAG,
    POSTAL_CODE_MISSING_FLAG,
    AGE_MISSING_FLAG,
]


for feature_name in SELECTED_MISSINGNESS_FEATURES:
    assert set(
        df[feature_name].unique()
    ).issubset({0, 1}), (
        f"Unexpected values found in "
        f"{feature_name}."
    )

assert (
    df["feature_gender_unknown_flag"]
    .eq(
        df[GENDER_FEATURE_COLUMN]
        .eq("Unknown")
        .astype("int8")
    )
    .all()
), (
    "Gender unknown flag does not match "
    "the standardized gender category."
)




assert (
    df.loc[
        df[GENDER_MISSING_FLAG].eq(1),
        "feature_gender_unknown_flag",
    ]
    .eq(1)
    .all()
), (
    "Some missing gender records were not "
    "classified as Unknown."
)

assert (
    df[POSTAL_CODE_MISSING_FLAG]
    .eq(
        df[POSTAL_CODE_FEATURE]
        .isna()
        .astype("int8")
    )
    .all()
), (
    "Postal-code missing flag does not match "
    "postal-code missingness."
)



assert (
    df[AGE_MISSING_FLAG]
    .eq(
        df[AGE_SOURCE_COLUMN]
        .isna()
        .astype("int8")
    )
    .all()
), (
    "Age missing flag does not match "
    "donor-age missingness."
)

In [100]:
# Summarize missingness features
missingness_feature_summary = pd.DataFrame({
    "feature_name": SELECTED_MISSINGNESS_FEATURES,
})

missingness_feature_summary["flagged_records"] = (
    missingness_feature_summary["feature_name"]
    .map(lambda column: df[column].sum())
)

missingness_feature_summary["percentage"] = (
    missingness_feature_summary["flagged_records"]
    .div(len(df))
    .mul(100)
)

missingness_feature_summary["unique_values"] = (
    missingness_feature_summary["feature_name"]
    .map(lambda column: df[column].nunique())
)

missingness_feature_summary["zero_variance"] = (
    missingness_feature_summary["unique_values"]
    .eq(1)
)

display(missingness_feature_summary.style
    .hide(axis="index")
    .set_properties(**{"text-align": "left"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "left")],
    }])
    .format({
        "flagged_records": "{:,.0f}",
        "percentage": "{:,.2f}%",
        "unique_values": "{:,.0f}",
    })
)

print(
    "\nSelected missingness feature "
    "validation passed."
)

feature_name,flagged_records,percentage,unique_values,zero_variance
feature_gender_unknown_flag,"1,594",4.63%,2,False
feature_gender_missing_flag,491,1.43%,2,False
feature_preferred_address_missing_flag,"4,033",11.72%,2,False
feature_postal_code_missing_flag,91,0.26%,2,False
feature_age_missing_flag,0,0.00%,1,True



Selected missingness feature validation passed.


## Missingness Features

Missingness indicators were retained only where missing or unknown values may reflect a meaningful donor characteristic or data-collection pattern.
- `feature_gender_unknown_flag` identifies 1,594 records (4.63%) categorized as `Unknown`.
- `feature_gender_missing_flag` identifies the 491 records (1.43%) whose original gender value was missing.
- `feature_preferred_address_missing_flag` identifies 4,033 records (11.72%) with no preferred address type.
- `feature_postal_code_missing_flag` identifies 91 records (0.26%) with no postal code.
- `feature_age_missing_flag` contains no flagged records because `donor_age` has no missing values.

The gender indicators represent different conditions. `feature_gender_unknown_flag` includes all standardized unknown values, while `feature_gender_missing_flag` identifies only values that were originally missing.

All selected missingness features passed binary and source-alignment validation. `feature_age_missing_flag` has zero variance and should be excluded during Phase 5 preprocessing unless future data introduces missing age values.

No additional missingness indicators were created because missingness flags should only be added when they have a meaningful interpretation rather than generated automatically for every feature.


In [101]:
# Create redundancy groups
def get_existing_columns(columns):
    return [
        column
        for column in columns
        if column in df.columns
    ]

TOTAL_DONATION_FEATURE = (
    "feature_past_5yr_total_donation"
)

AVERAGE_DONATION_FEATURE = (
    "feature_past_5yr_average_donation"
)

YEARS_DONATED_FEATURE = (
    "feature_years_donated_past_5yr"
)

DONATION_FREQUENCY_FEATURE = (
    "feature_past_5yr_donation_frequency_rate"
)


DONATION_RECENCY_FEATURE = (
    "feature_years_since_last_donation_past_5yr"
)

RECENCY_INDICATOR_KEYWORDS = [
    "donated_last",
    "within_2",
    "recent_donor",
    "lapsed_donor",
    "never_donated",
]

RECENCY_INDICATOR_FEATURES = [
    column
    for column in df.columns
    if (
        column.startswith("feature_")
        and any(
            keyword in column
            for keyword in RECENCY_INDICATOR_KEYWORDS
        )
        and set(
            df[column]
            .dropna()
            .unique()
        ).issubset({0, 1})
    )
]

print(
    "Recency indicator features identified:"
)

for column in RECENCY_INDICATOR_FEATURES:
    print(f"- {column}")

assert RECENCY_INDICATOR_FEATURES, (
    "No binary donation recency indicators "
    "were identified."
)


ORIGINAL_ENGAGEMENT_FEATURES = [
    "is_alumnus_flag",
    "is_parent_flag",
    "has_involvement_flag",
    "has_email_flag",
]


ENGAGEMENT_SCORE_COMPONENTS = {
    "feature_engagement_score": [
        "is_alumnus_flag",
        "is_parent_flag",
        "has_involvement_flag",
        "has_email_flag",
    ],
    "feature_core_engagement_score": [
        "is_alumnus_flag",
        "has_involvement_flag",
        "has_email_flag",
    ],
    "feature_expanded_engagement_score": [
        "is_alumnus_flag",
        "is_parent_flag",
        "has_involvement_flag",
        "has_email_flag",
    ],
}

ENGAGEMENT_SCORE_COMPONENTS = {
    score: components
    for score, components
    in ENGAGEMENT_SCORE_COMPONENTS.items()
    if score in df.columns
}


DONATION_LOG_FEATURE_PAIRS = [
    (
        "feature_past_5yr_total_donation",
        "feature_log_past_5yr_total_donation",
    ),
    (
        "feature_past_5yr_average_donation",
        "feature_log_past_5yr_average_donation",
    ),
    (
        "feature_past_5yr_max_donation",
        "feature_log_past_5yr_max_donation",
    ),
    (
        "feature_most_recent_positive_donation",
        "feature_log_most_recent_positive_donation",
    ),
]

DONATION_LOG_FEATURE_PAIRS = [
    (original, transformed)
    for original, transformed
    in DONATION_LOG_FEATURE_PAIRS
    if (
        original in df.columns
        and transformed in df.columns
    )
]


required_redundancy_columns = [
    TOTAL_DONATION_FEATURE,
    AVERAGE_DONATION_FEATURE,
    YEARS_DONATED_FEATURE,
    DONATION_FREQUENCY_FEATURE,
    DONATION_RECENCY_FEATURE,
    *ORIGINAL_ENGAGEMENT_FEATURES,
]

missing_redundancy_columns = [
    column
    for column in required_redundancy_columns
    if column not in df.columns
]

assert not missing_redundancy_columns, (
    "Required redundancy-review columns are "
    f"missing: {missing_redundancy_columns}"
)

assert RECENCY_INDICATOR_FEATURES, (
    "No donation recency indicators were found."
)

assert ENGAGEMENT_SCORE_COMPONENTS, (
    "No engagement score features were found."
)

assert DONATION_LOG_FEATURE_PAIRS, (
    "No original and log-transformed donation "
    "feature pairs were found."
)

Recency indicator features identified:
- feature_donated_last_year_flag
- feature_donated_within_2_years_flag
- feature_lapsed_donor_flag
- feature_never_donated_past_5yr_flag


In [102]:
# Create pariwise relationships
redundancy_pairs = []


redundancy_pairs.append({
    "feature_family": "Donation aggregates",
    "feature_1": TOTAL_DONATION_FEATURE,
    "feature_2": AVERAGE_DONATION_FEATURE,
})



redundancy_pairs.append({
    "feature_family": "Donation frequency",
    "feature_1": YEARS_DONATED_FEATURE,
    "feature_2": DONATION_FREQUENCY_FEATURE,
})


for indicator_feature in RECENCY_INDICATOR_FEATURES:
    redundancy_pairs.append({
        "feature_family": "Donation recency",
        "feature_1": DONATION_RECENCY_FEATURE,
        "feature_2": indicator_feature,
    })


for score_feature, component_features in (
    ENGAGEMENT_SCORE_COMPONENTS.items()
):
    for component_feature in component_features:
        redundancy_pairs.append({
            "feature_family": "Engagement",
            "feature_1": score_feature,
            "feature_2": component_feature,
        })


for original_feature, log_feature in (
    DONATION_LOG_FEATURE_PAIRS
):
    redundancy_pairs.append({
        "feature_family": "Donation transformation",
        "feature_1": original_feature,
        "feature_2": log_feature,
    })


redundancy_review = pd.DataFrame(
    redundancy_pairs
)


def calculate_pair_correlation(row, method):
    pair_data = df[
        [row["feature_1"], row["feature_2"]]
    ].dropna()

    return pair_data[
        row["feature_1"]
    ].corr(
        pair_data[row["feature_2"]],
        method=method,
    )


redundancy_review["pearson_correlation"] = (
    redundancy_review.apply(
        calculate_pair_correlation,
        axis=1,
        method="pearson",
    )
)

redundancy_review["spearman_correlation"] = (
    redundancy_review.apply(
        calculate_pair_correlation,
        axis=1,
        method="spearman",
    )
)


redundancy_review["maximum_absolute_correlation"] = (
    redundancy_review[
        [
            "pearson_correlation",
            "spearman_correlation",
        ]
    ]
    .abs()
    .max(axis=1)
)


def classify_relationship(correlation):
    if correlation >= 0.90:
        return "Very strong"
    if correlation >= 0.70:
        return "Strong"
    if correlation >= 0.40:
        return "Moderate"
    return "Weak"


redundancy_review["relationship_strength"] = (
    redundancy_review[
        "maximum_absolute_correlation"
    ]
    .map(classify_relationship)
)


display(redundancy_review.style
    .hide(axis="index")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "center")],
    }])
    .format({
        "pearson_correlation": "{:,.4f}",
        "spearman_correlation": "{:,.4f}",
        "maximum_absolute_correlation": "{:,.4f}",
    })
)

feature_family,feature_1,feature_2,pearson_correlation,spearman_correlation,maximum_absolute_correlation,relationship_strength
Donation aggregates,feature_past_5yr_total_donation,feature_past_5yr_average_donation,1.0000,1.0000,1.0000,Very strong
Donation frequency,feature_years_donated_past_5yr,feature_past_5yr_donation_frequency_rate,1.0000,1.0000,1.0000,Very strong
Donation recency,feature_years_since_last_donation_past_5yr,feature_donated_last_year_flag,-0.6528,-0.5647,0.6528,Moderate
Donation recency,feature_years_since_last_donation_past_5yr,feature_donated_within_2_years_flag,-0.8506,-0.7604,0.8506,Strong
Donation recency,feature_years_since_last_donation_past_5yr,feature_lapsed_donor_flag,-0.3618,-0.5213,0.5213,Moderate
Donation recency,feature_years_since_last_donation_past_5yr,feature_never_donated_past_5yr_flag,0.9329,0.9853,0.9853,Very strong
Engagement,feature_engagement_score,is_alumnus_flag,0.7272,0.6883,0.7272,Strong
Engagement,feature_engagement_score,is_parent_flag,0.2813,0.3030,0.3030,Weak
Engagement,feature_engagement_score,has_involvement_flag,0.7119,0.6649,0.7119,Strong
Engagement,feature_engagement_score,has_email_flag,0.7641,0.7661,0.7661,Strong


In [103]:
# Validate deterministic relationships
deterministic_relationships = []


def add_relationship_check(
    feature_family,
    relationship,
    confirmed,
):
    deterministic_relationships.append({
        "feature_family": feature_family,
        "relationship": relationship,
        "confirmed": bool(confirmed),
    })


add_relationship_check(
    feature_family="Donation aggregates",
    relationship=(
        f"{AVERAGE_DONATION_FEATURE} = "
        f"{TOTAL_DONATION_FEATURE} / 5"
    ),
    confirmed=np.allclose(
        df[AVERAGE_DONATION_FEATURE],
        df[TOTAL_DONATION_FEATURE] / 5,
        equal_nan=True,
    ),
)

add_relationship_check(
    feature_family="Donation frequency",
    relationship=(
        f"{DONATION_FREQUENCY_FEATURE} = "
        f"{YEARS_DONATED_FEATURE} / 5"
    ),
    confirmed=np.allclose(
        df[DONATION_FREQUENCY_FEATURE],
        df[YEARS_DONATED_FEATURE] / 5,
        equal_nan=True,
    ),
)



for score_feature, component_features in (
    ENGAGEMENT_SCORE_COMPONENTS.items()
):
    expected_score = (
        df[component_features]
        .sum(axis=1)
    )

    add_relationship_check(
        feature_family="Engagement",
        relationship=(
            f"{score_feature} = sum of "
            f"{', '.join(component_features)}"
        ),
        confirmed=np.allclose(
            df[score_feature],
            expected_score,
            equal_nan=True,
        ),
    )


for original_feature, log_feature in (
    DONATION_LOG_FEATURE_PAIRS
):
    add_relationship_check(
        feature_family="Donation transformation",
        relationship=(
            f"{log_feature} = "
            f"log1p({original_feature})"
        ),
        confirmed=np.allclose(
            df[log_feature],
            np.log1p(df[original_feature]),
            equal_nan=True,
        ),
    )


deterministic_relationship_summary = (
    pd.DataFrame(
        deterministic_relationships
    )
)


display(deterministic_relationship_summary.style
    .hide(axis="index")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "center")],
    }])
)


assert (
    deterministic_relationship_summary[
        "confirmed"
    ]
    .all()
), (
    "At least one deterministic feature "
    "relationship did not match its definition."
)

print(
    "\nDeterministic redundancy checks passed."
)

feature_family,relationship,confirmed
Donation aggregates,feature_past_5yr_average_donation = feature_past_5yr_total_donation / 5,True
Donation frequency,feature_past_5yr_donation_frequency_rate = feature_years_donated_past_5yr / 5,True
Engagement,"feature_engagement_score = sum of is_alumnus_flag, is_parent_flag, has_involvement_flag, has_email_flag",True
Engagement,"feature_core_engagement_score = sum of is_alumnus_flag, has_involvement_flag, has_email_flag",True
Donation transformation,feature_log_past_5yr_total_donation = log1p(feature_past_5yr_total_donation),True
Donation transformation,feature_log_past_5yr_average_donation = log1p(feature_past_5yr_average_donation),True
Donation transformation,feature_log_past_5yr_max_donation = log1p(feature_past_5yr_max_donation),True
Donation transformation,feature_log_most_recent_positive_donation = log1p(feature_most_recent_positive_donation),True



Deterministic redundancy checks passed.


In [104]:
# Find duplicate features
duplicate_check_features = get_existing_columns([
    *RECENCY_INDICATOR_FEATURES,
    *ENGAGEMENT_SCORE_COMPONENTS.keys(),
    "feature_has_any_engagement",
    "feature_high_engagement",
    "feature_core_high_engagement_flag",
    "feature_expanded_high_engagement_flag",
    *ORIGINAL_ENGAGEMENT_FEATURES,
])


duplicate_feature_pairs = []


for index, feature_1 in enumerate(
    duplicate_check_features
):
    for feature_2 in duplicate_check_features[
        index + 1:
    ]:
        exact_match = (
            df[feature_1]
            .eq(df[feature_2])
            .all()
        )

        binary_features = (
            set(df[feature_1].dropna().unique())
            .issubset({0, 1})
            and
            set(df[feature_2].dropna().unique())
            .issubset({0, 1})
        )

        inverse_match = (
            binary_features
            and df[feature_1]
            .eq(1 - df[feature_2])
            .all()
        )

        if exact_match or inverse_match:
            duplicate_feature_pairs.append({
                "feature_1": feature_1,
                "feature_2": feature_2,
                "relationship": (
                    "Exact duplicate"
                    if exact_match
                    else "Exact inverse"
                ),
            })


duplicate_feature_summary = pd.DataFrame(
    duplicate_feature_pairs,
    columns=[
        "feature_1",
        "feature_2",
        "relationship",
    ],
)


if duplicate_feature_summary.empty:
    print(
        "No exact duplicate or inverse "
        "features were found."
    )
else:
    display(duplicate_feature_summary.style
        .hide(axis="index")
        .set_properties(**{"text-align": "center"})
        .set_table_styles([{
            "selector": "th",
            "props": [("text-align", "center")],
        }])
    )


print(
    "Redundant features were reviewed. "
    "No features were removed."
)

No exact duplicate or inverse features were found.
Redundant features were reviewed. No features were removed.


## Redundant Feature Review

Related engineered features were reviewed using correlation, formula checks, and duplicate checks. No features were removed during this phase.

### Donation Aggregates and Frequency
- `feature_past_5yr_total_donation` and `feature_past_5yr_average_donation` are perfectly correlated because the average equals the total divided by five.
- `feature_years_donated_past_5yr` and `feature_past_5yr_donation_frequency_rate` are perfectly correlated because the frequency rate equals years donated divided by five.

### Donation Recency
- `feature_donated_within_2_years_flag` has a strong negative relationship with numeric recency.
- `feature_donated_last_year_flag` and `feature_lapsed_donor_flag` have moderate relationships with numeric recency.
- `feature_never_donated_past_5yr_flag` has a very strong positive relationship with numeric recency because records with no historical donations receive the highest recency value.

### Engagement Features

- `feature_engagement_score` is strongly related to the alumnus, involvement, and email flags, but weakly related to the parent flag.
- `feature_core_engagement_score` is strongly related to each of its component flags.
- Phase 5 should compare the combined scores with the individual engagement features.

### Original and Log-Transformed Donations
- Log-transformed donation features have perfect Spearman correlations with their original values because `log1p()` preserves ranking.
- Pearson correlations are weak because the transformation compresses the highly skewed donation values.

All deterministic relationships matched their definitions, and no exact duplicate or inverse features were found.
